
# CELDA 1 — Configuración general del HBM v2 para PM2.5

# **Objetivo:**
dejar definidas las rutas, parámetros y nombres de columnas que usará
el HBM v2 con observaciones reales de estaciones.

# **Qué hace esta celda:**
 1. Define la carpeta de trabajo y las rutas de entrada.
 2. Declara el contaminante objetivo (`PM25`).
 3. Define las covariables base del HBM:
   - `Vel_viento_idw`
   - `diff_PM25_grid`
   - `grad_PM25_grid`
    - `adv_proxy_PM25_grid`
 4. Crea la carpeta de salida donde se guardarán paneles, folds,
    resultados de validación y superficies finales.

 **Nota metodológica importante:**
 En esta versión no se usa `PM25_idw` como predictor directo del HBM.
 La respuesta será la observación real de estación (`PM25_obs`).


In [1]:
# %%
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# =========================
# 0) Rutas
# =========================
PROJECT_ROOT = Path.cwd()

# Ajusta esta línea si tus archivos están en otra carpeta
DATA_DIR = PROJECT_ROOT

OBS_CSV  = DATA_DIR / "panel_ambiental_mensual_2020_2024_coords_corregidas.csv"
GRID_CSV = DATA_DIR / "grid_3km_AD_mensual_2020_2024.csv"
W_CSV    = DATA_DIR / "W_grid_3km_queen.csv"

OUT_DIR = PROJECT_ROOT / "HBM_PM25_V2_OUT"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# 1) Configuración HBM v2
# =========================
POLL = "PM25"
OBS_COL = "PM25"
VALID_COL = "valido_PM25"

X_COLS_RAW = [
    "Vel_viento_idw",
    "diff_PM25_grid",
    "grad_PM25_grid",
    "adv_proxy_PM25_grid",
]

USE_LOG = True
EPS = 1e-6

# folds temporales
FOLDS = {
    "fold_1": {"train_years": [2020, 2021], "test_years": [2022]},
    "fold_2": {"train_years": [2020, 2021, 2022], "test_years": [2023]},
    "fold_3": {"train_years": [2020, 2021, 2022, 2023], "test_years": [2024]},
}

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR     :", DATA_DIR)
print("OUT_DIR      :", OUT_DIR)
print("\nArchivos de entrada:")
for p in [OBS_CSV, GRID_CSV, W_CSV]:
    print(" -", p.name, "| exists:", p.exists())

for p in [OBS_CSV, GRID_CSV, W_CSV]:
    if not p.exists():
        raise FileNotFoundError(f"No encuentro el archivo: {p}")

print("\nConfiguración lista.")
print("POLL         :", POLL)
print("OBS_COL      :", OBS_COL)
print("VALID_COL    :", VALID_COL)
print("X_COLS_RAW   :", X_COLS_RAW)
print("USE_LOG      :", USE_LOG)


PROJECT_ROOT: d:\TRABAJO DE GRADO BEN-MAP\CODIGO
DATA_DIR     : d:\TRABAJO DE GRADO BEN-MAP\CODIGO
OUT_DIR      : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT

Archivos de entrada:
 - panel_ambiental_mensual_2020_2024_coords_corregidas.csv | exists: True
 - grid_3km_AD_mensual_2020_2024.csv | exists: True
 - W_grid_3km_queen.csv | exists: True

Configuración lista.
POLL         : PM25
OBS_COL      : PM25
VALID_COL    : valido_PM25
X_COLS_RAW   : ['Vel_viento_idw', 'diff_PM25_grid', 'grad_PM25_grid', 'adv_proxy_PM25_grid']
USE_LOG      : True



# CELDA 2 — Cargar, limpiar y construir el panel base de modelación

# **Objetivo:**
# construir dos paneles:

 1. `grid_base`:
    la superficie completa celda–mes de la malla 3 km con covariables A–D.

 2. `obs_panel`:
    las observaciones reales de estación para PM2.5, ya enlazadas a:
   - `cell_id`
    - `fecha`
    - covariables de la malla

 **Qué hace esta celda:**
 1. Carga el panel de estaciones corregido.
 2. Carga la malla 3 km con términos A–D.
 3. Convierte fechas a formato mensual.
 4. Filtra observaciones válidas de PM2.5.
 5. Une cada observación con la covariable de su celda y mes.
 6. Crea índices globales:
    - `cell_idx`
    - `time_idx`
 7. Guarda paneles limpios para el HBM.


In [2]:
# %%
# =========================
# 2) Cargar archivos
# =========================
obs = pd.read_csv(OBS_CSV)
grid = pd.read_csv(GRID_CSV)
w = pd.read_csv(W_CSV)

# =========================
# 3) Fechas
# =========================
obs["fecha"] = pd.to_datetime(
    dict(year=obs["Año"].astype(int), month=obs["Mes"].astype(int), day=1),
    errors="coerce"
)
grid["fecha"] = pd.to_datetime(grid["fecha"], errors="coerce")

if obs["fecha"].isna().any():
    raise ValueError("Hay fechas inválidas en el panel de estaciones.")
if grid["fecha"].isna().any():
    raise ValueError("Hay fechas inválidas en la malla 3km.")

# =========================
# 4) Validación básica de columnas
# =========================
obs_needed = ["Estacion", "cell_id", "fecha", OBS_COL, VALID_COL]
grid_needed = ["cell_id", "fecha"] + X_COLS_RAW

for c in obs_needed:
    if c not in obs.columns:
        raise ValueError(f"Falta la columna '{c}' en el panel de estaciones.")

for c in grid_needed:
    if c not in grid.columns:
        raise ValueError(f"Falta la columna '{c}' en la malla 3km.")

# =========================
# 5) Filtrar observaciones válidas PM2.5
# =========================
obs[VALID_COL] = obs[VALID_COL].astype(bool)

obs_pm = obs.loc[
    (obs[VALID_COL] == True) &
    (obs[OBS_COL].notna()) &
    (obs["cell_id"].notna())
].copy()

obs_pm = obs_pm.rename(columns={OBS_COL: "PM25_obs"})

# =========================
# 6) Construir grid_base
# =========================
grid_base = grid[["cell_id", "fecha"] + X_COLS_RAW].copy()
grid_base["year"] = grid_base["fecha"].dt.year
grid_base["month"] = grid_base["fecha"].dt.month

# =========================
# 7) Unir observaciones con covariables de malla
# =========================
obs_panel = obs_pm.merge(
    grid_base,
    on=["cell_id", "fecha"],
    how="left",
    validate="many_to_one"
)

missing_merge = obs_panel[X_COLS_RAW].isna().any(axis=1).sum()
if missing_merge > 0:
    raise ValueError(
        f"Hay {missing_merge} observaciones sin covariables de la malla. "
        "Revisa cell_id y fecha."
    )

obs_panel["year"] = obs_panel["fecha"].dt.year
obs_panel["month"] = obs_panel["fecha"].dt.month

# =========================
# 8) Índices globales de celda y tiempo
# =========================
cell_ids = np.sort(grid_base["cell_id"].unique())
time_ids = np.sort(grid_base["fecha"].unique())

cell_map = {cid: i for i, cid in enumerate(cell_ids)}
time_map = {tt: j for j, tt in enumerate(time_ids)}

grid_base["cell_idx"] = grid_base["cell_id"].map(cell_map).astype(int)
grid_base["time_idx"] = grid_base["fecha"].map(time_map).astype(int)

obs_panel["cell_idx"] = obs_panel["cell_id"].map(cell_map).astype(int)
obs_panel["time_idx"] = obs_panel["fecha"].map(time_map).astype(int)

# =========================
# 9) Guardar paneles base
# =========================
grid_base_path = OUT_DIR / "HBM_PM25_grid_base_v2.csv"
obs_panel_path = OUT_DIR / "HBM_PM25_obs_panel_v2.csv"

grid_base.to_csv(grid_base_path, index=False)
obs_panel.to_csv(obs_panel_path, index=False)

# =========================
# 10) Resumen
# =========================
print("Resumen del panel base HBM v2")
print("- obs_panel filas          :", len(obs_panel))
print("- estaciones únicas       :", obs_panel["Estacion"].nunique())
print("- celdas con observación   :", obs_panel["cell_id"].nunique())
print("- grid_base filas          :", len(grid_base))
print("- celdas totales en malla  :", len(cell_ids))
print("- meses totales            :", len(time_ids))
print("\nArchivos guardados:")
print(" -", grid_base_path)
print(" -", obs_panel_path)


Resumen del panel base HBM v2
- obs_panel filas          : 808
- estaciones únicas       : 16
- celdas con observación   : 14
- grid_base filas          : 15240
- celdas totales en malla  : 254
- meses totales            : 60

Archivos guardados:
 - d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_grid_base_v2.csv
 - d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_obs_panel_v2.csv



# # CELDA 3 — Preparar vecindad espacial y folds temporales

# **Objetivo:**
# dejar lista la estructura espacial del HBM y la validación temporal.

# **Qué hace esta celda:**
 1. Toma `W_grid_3km_queen.csv`.
 2. Verifica que las celdas de la vecindad existan en la malla.
 3. Convierte `cell_id` y `neighbor_id` a índices internos `i`, `j`.
 4. Elimina duplicados dirigidos para dejar pares únicos no dirigidos.
 5. Guarda la vecindad final para el modelo.
 6. Guarda también la definición de folds temporales.

# **Salidas:**
# - `HBM_PM25_W_edges_queen_v2.csv`
# - `HBM_PM25_folds_v2.json`

In [3]:
# %%
# =========================
# CELDA 3) Vecindad espacial y folds temporales
# =========================

# Validación mínima de columnas en W
if not {"cell_id", "neighbor_id"}.issubset(w.columns):
    raise ValueError("W_grid_3km_queen.csv debe tener las columnas: 'cell_id' y 'neighbor_id'.")

# Filtrar solo relaciones cuyos nodos existan en la malla
w_use = w[
    w["cell_id"].isin(cell_ids) &
    w["neighbor_id"].isin(cell_ids)
].copy()

if len(w_use) == 0:
    raise ValueError("La vecindad quedó vacía después de filtrar por celdas de la malla.")

# Mapear a índices internos
w_use["i"] = w_use["cell_id"].map(cell_map)
w_use["j"] = w_use["neighbor_id"].map(cell_map)

if w_use["i"].isna().any() or w_use["j"].isna().any():
    raise ValueError("Hay relaciones de vecindad que no pudieron mapearse a índices internos.")

w_use["i"] = w_use["i"].astype(int)
w_use["j"] = w_use["j"].astype(int)

# Quitar lazos propios si existieran
w_use = w_use.loc[w_use["i"] != w_use["j"]].copy()

# Dejar pares únicos no dirigidos
ii = np.minimum(w_use["i"].values, w_use["j"].values)
jj = np.maximum(w_use["i"].values, w_use["j"].values)

pairs = np.unique(np.column_stack([ii, jj]), axis=0)
w_edges = pd.DataFrame(pairs, columns=["i", "j"])

# Guardar W final
w_edges_path = OUT_DIR / "HBM_PM25_W_edges_queen_v2.csv"
w_edges.to_csv(w_edges_path, index=False)

# Guardar folds
folds_path = OUT_DIR / "HBM_PM25_folds_v2.json"
with open(folds_path, "w", encoding="utf-8") as f:
    json.dump(FOLDS, f, ensure_ascii=False, indent=2)

# Resumen
print("Resumen CELDA 3")
print("- relaciones originales en W      :", len(w))
print("- relaciones válidas tras filtro  :", len(w_use))
print("- edges únicos no dirigidos       :", len(w_edges))
print("- nodos únicos en W               :", len(set(w_use['i']).union(set(w_use['j']))))

print("\nArchivos guardados:")
print("-", w_edges_path)
print("-", folds_path)

print("\nFolds temporales:")
print(json.dumps(FOLDS, ensure_ascii=False, indent=2))

Resumen CELDA 3
- relaciones originales en W      : 1610
- relaciones válidas tras filtro  : 1610
- edges únicos no dirigidos       : 805
- nodos únicos en W               : 254

Archivos guardados:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_W_edges_queen_v2.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_folds_v2.json

Folds temporales:
{
  "fold_1": {
    "train_years": [
      2020,
      2021
    ],
    "test_years": [
      2022
    ]
  },
  "fold_2": {
    "train_years": [
      2020,
      2021,
      2022
    ],
    "test_years": [
      2023
    ]
  },
  "fold_3": {
    "train_years": [
      2020,
      2021,
      2022,
      2023
    ],
    "test_years": [
      2024
    ]
  }
}



# # CELDA 4 — Funciones auxiliares para escalamiento, métricas e intervalos

# **Objetivo:**
definir funciones reutilizables para preparar los datos del HBM v2
y evaluar sus predicciones de manera consistente.

**Qué hace esta celda:**
 1. Define una función para estandarizar covariables usando solo los años de entrenamiento.
 2. Define una función para llevar esas covariables escaladas al panel observado.
 3. Define una función para calcular métricas puntuales:
    - MAE
    - RMSE
    - sesgo
    - correlación
    - R²
 4. Define métricas probabilísticas:
    - cobertura del intervalo 90%
    - ancho promedio del intervalo
    - WIS
 5. Define una función para extraer `p05`, `p50` y `p95`
    a partir del posterior del modelo.

 **Importancia metodológica:**
 el escalamiento se hace usando únicamente el período de entrenamiento,
 para evitar fuga de información hacia los folds de prueba.

In [4]:
# %%
# =========================
# CELDA 4) Funciones auxiliares
# =========================

def scale_grid_by_train_years(grid_df, x_cols, train_years):
    """
    Estandariza covariables usando solo las filas de grid_df
    pertenecientes a los años de entrenamiento.

    Parámetros
    ----------
    grid_df : pd.DataFrame
        Panel celda-mes de la malla.
    x_cols : list[str]
        Lista de covariables crudas a escalar.
    train_years : list[int]
        Años que pertenecen al conjunto de entrenamiento.

    Retorna
    -------
    grid_scaled : pd.DataFrame
        DataFrame con nuevas columnas *_z.
    params : dict
        Media y desviación estándar usadas para cada covariable.
    """
    grid_scaled = grid_df.copy()
    params = {}

    train_mask = grid_scaled["year"].isin(train_years)

    for col in x_cols:
        mu = grid_scaled.loc[train_mask, col].mean()
        sd = grid_scaled.loc[train_mask, col].std(ddof=0)

        if pd.isna(sd) or sd == 0:
            sd = 1.0

        z_col = f"{col}_z"
        grid_scaled[z_col] = (grid_scaled[col] - mu) / sd
        params[col] = {"mu": float(mu), "sd": float(sd)}

    return grid_scaled, params


def merge_scaled_covariates_to_obs(obs_df, grid_scaled, x_cols):
    """
    Lleva las covariables estandarizadas desde la malla al panel observado,
    usando la llave (cell_id, fecha).
    """
    z_cols = [f"{c}_z" for c in x_cols]

    out = obs_df.drop(columns=x_cols, errors="ignore").merge(
        grid_scaled[["cell_id", "fecha"] + z_cols],
        on=["cell_id", "fecha"],
        how="left",
        validate="many_to_one"
    )

    missing = out[z_cols].isna().any(axis=1).sum()
    if missing > 0:
        raise ValueError(f"Hay {missing} observaciones sin covariables escaladas.")

    return out


def interval_metrics(y_true, p05, p50, p95, alpha=0.10):
    """
    Calcula métricas puntuales y probabilísticas
    sobre un intervalo central del 90%.
    """
    y_true = np.asarray(y_true, dtype=float)
    p05 = np.asarray(p05, dtype=float)
    p50 = np.asarray(p50, dtype=float)
    p95 = np.asarray(p95, dtype=float)

    m = ~np.isnan(y_true) & ~np.isnan(p05) & ~np.isnan(p50) & ~np.isnan(p95)
    y_true = y_true[m]
    p05 = p05[m]
    p50 = p50[m]
    p95 = p95[m]

    if len(y_true) == 0:
        return {
            "n": 0,
            "mae": np.nan,
            "rmse": np.nan,
            "bias": np.nan,
            "r": np.nan,
            "r2": np.nan,
            "coverage_90": np.nan,
            "width_90_mean": np.nan,
            "wis_90": np.nan,
        }

    err = p50 - y_true
    mae = np.mean(np.abs(err))
    rmse = np.sqrt(np.mean(err**2))
    bias = np.mean(err)

    if np.std(y_true) == 0 or np.std(p50) == 0:
        r = np.nan
        r2 = np.nan
    else:
        r = np.corrcoef(y_true, p50)[0, 1]
        r2 = r**2

    coverage = np.mean((y_true >= p05) & (y_true <= p95))
    width = np.mean(p95 - p05)

    wis = np.mean(
        (p95 - p05)
        + (2 / alpha) * (p05 - y_true) * (y_true < p05)
        + (2 / alpha) * (y_true - p95) * (y_true > p95)
    )

    return {
        "n": int(len(y_true)),
        "mae": float(mae),
        "rmse": float(rmse),
        "bias": float(bias),
        "r": float(r) if not np.isnan(r) else np.nan,
        "r2": float(r2) if not np.isnan(r2) else np.nan,
        "coverage_90": float(coverage),
        "width_90_mean": float(width),
        "wis_90": float(wis),
    }


def posterior_predict_concentration(idata, X_mat, cell_idx_arr, time_idx_arr, eps=1e-6):
    """
    A partir del posterior del HBM, obtiene p05, p50 y p95
    en escala original de concentración.
    """
    alpha_s = idata.posterior["alpha"].values.reshape(-1)
    beta_s = idata.posterior["beta"].values.reshape(-1, idata.posterior["beta"].values.shape[-1])
    phi_s = idata.posterior["phi"].values.reshape(-1, idata.posterior["phi"].values.shape[-1])
    delta_s = idata.posterior["delta"].values.reshape(-1, idata.posterior["delta"].values.shape[-1])

    mu_s = (
        alpha_s[:, None]
        + (beta_s @ X_mat.T)
        + phi_s[:, cell_idx_arr]
        + delta_s[:, time_idx_arr]
    )

    c_s = np.exp(mu_s) - eps

    p05 = np.quantile(c_s, 0.05, axis=0)
    p50 = np.quantile(c_s, 0.50, axis=0)
    p95 = np.quantile(c_s, 0.95, axis=0)

    return p05, p50, p95


print("CELDA 4 cargada correctamente.")
print("Funciones disponibles:")
print("- scale_grid_by_train_years")
print("- merge_scaled_covariates_to_obs")
print("- interval_metrics")
print("- posterior_predict_concentration")

CELDA 4 cargada correctamente.
Funciones disponibles:
- scale_grid_by_train_years
- merge_scaled_covariates_to_obs
- interval_metrics
- posterior_predict_concentration



 # CELDA 5 — Definición del modelo base M1: ICAR + RW1

 **Objetivo:**
 dejar definida la función que ajusta el HBM base para un fold temporal.

 **Especificación del modelo M1:**

 - Respuesta observada:
   `PM25_obs`
 - Escala:
   logarítmica
 - Efectos fijos:
   covariables A–D + viento
 - Efecto espacial:
   `ICAR`
 - Efecto temporal:
   `RW1`

 **Forma general del modelo:**

 `log(PM25_obs) = alpha + X beta + phi_i + delta_t + error`

 donde:
 - `phi_i` representa el efecto espacial estructurado sobre la malla
 - `delta_t` representa la evolución temporal mensual

 **Qué hace esta celda:**
 1. Importa PyMC, PyTensor y ArviZ.
 2. Define la función `fit_hbm_m1_fold`.
 3. La función:
    - recibe train/test por años,
    - ajusta el modelo bayesiano,
    - predice sobre el período test,
    - calcula percentiles posteriores,
    - devuelve métricas e intervalos.

 **Nota:**
 si PyMC no está instalado en tu entorno, esta celda te lo indicará.

In [5]:
# %%
# =========================
# CELDA 5) Modelo base M1: ICAR + RW1
# =========================

try:
    import pymc as pm
    import pytensor.tensor as pt
    import arviz as az
except Exception as e:
    raise ImportError(
        "No se pudieron importar pymc / pytensor / arviz.\n"
        "Instala en tu entorno:\n"
        "pip install pymc arviz pytensor\n\n"
        f"Detalle original: {e}"
    )


def fit_hbm_m1_fold(
    obs_scaled,
    grid_scaled,
    w_edges_df,
    x_cols_raw,
    train_years,
    test_years,
    n_cells,
    n_times,
    draws=1000,
    tune=1000,
    chains=4,
    target_accept=0.95,
    random_seed=42,
):
    """
    Ajusta el HBM base M1 para un fold temporal:
    - espacial ICAR
    - temporal RW1
    - respuesta observada PM25_obs
    """

    z_cols = [f"{c}_z" for c in x_cols_raw]

    train_mask_obs = obs_scaled["year"].isin(train_years)
    test_mask_obs = obs_scaled["year"].isin(test_years)

    train_obs = obs_scaled.loc[train_mask_obs].copy()
    test_obs = obs_scaled.loc[test_mask_obs].copy()

    if len(train_obs) == 0:
        raise ValueError("No hay observaciones de entrenamiento para este fold.")
    if len(test_obs) == 0:
        raise ValueError("No hay observaciones de prueba para este fold.")

    # Matrices y vectores del conjunto de entrenamiento
    X_train = train_obs[z_cols].to_numpy(dtype=float)
    y_train_raw = train_obs["PM25_obs"].to_numpy(dtype=float)
    y_train = np.log(y_train_raw + EPS) if USE_LOG else y_train_raw

    cell_train = train_obs["cell_idx"].to_numpy(dtype=int)
    time_train = train_obs["time_idx"].to_numpy(dtype=int)

    # Matrices y vectores del conjunto de prueba
    X_test = test_obs[z_cols].to_numpy(dtype=float)
    y_test = test_obs["PM25_obs"].to_numpy(dtype=float)
    cell_test = test_obs["cell_idx"].to_numpy(dtype=int)
    time_test = test_obs["time_idx"].to_numpy(dtype=int)

    # Edges espaciales
    ei = w_edges_df["i"].to_numpy(dtype=int)
    ej = w_edges_df["j"].to_numpy(dtype=int)

    p = X_train.shape[1]

    with pm.Model() as model:
        # -------------------------
        # Efectos fijos
        # -------------------------
        alpha = pm.Normal("alpha", mu=0.0, sigma=5.0)
        beta = pm.Normal("beta", mu=0.0, sigma=1.0, shape=p)

        # -------------------------
        # Error observacional
        # -------------------------
        sigma_y = pm.HalfNormal("sigma_y", sigma=1.0)

        # -------------------------
        # Efecto espacial ICAR
        # -------------------------
        tau_phi = pm.Exponential("tau_phi", 1.0)
        phi_raw = pm.Normal("phi_raw", mu=0.0, sigma=1.0, shape=n_cells)
        phi = pm.Deterministic("phi", phi_raw - pt.mean(phi_raw))

        pm.Potential(
            "icar_penalty",
            -0.5 * tau_phi * pt.sum((phi[ei] - phi[ej]) ** 2)
        )

        # -------------------------
        # Efecto temporal RW1
        # -------------------------
        sigma_t = pm.HalfNormal("sigma_t", sigma=1.0)
        delta_raw = pm.GaussianRandomWalk("delta_raw", sigma=sigma_t, shape=n_times)
        delta = pm.Deterministic("delta", delta_raw - pt.mean(delta_raw))

        # -------------------------
        # Media del modelo
        # -------------------------
        mu_train = alpha + pt.dot(X_train, beta) + phi[cell_train] + delta[time_train]

        # -------------------------
        # Verosimilitud
        # -------------------------
        pm.Normal("y_obs", mu=mu_train, sigma=sigma_y, observed=y_train)

        # -------------------------
        # Muestreo MCMC
        # -------------------------
        idata = pm.sample(
            draws=draws,
            tune=tune,
            chains=chains,
            target_accept=target_accept,
            random_seed=random_seed,
            return_inferencedata=True,
            progressbar=True,
        )

    # -------------------------
    # Predicción sobre test
    # -------------------------
    p05_test, p50_test, p95_test = posterior_predict_concentration(
        idata=idata,
        X_mat=X_test,
        cell_idx_arr=cell_test,
        time_idx_arr=time_test,
        eps=EPS
    )

    test_pred = test_obs[
        ["Estacion", "fecha", "year", "month", "cell_id", "cell_idx", "time_idx", "PM25_obs"]
    ].copy()

    test_pred["p05_hbm"] = p05_test
    test_pred["p50_hbm"] = p50_test
    test_pred["p95_hbm"] = p95_test
    test_pred["width_90"] = test_pred["p95_hbm"] - test_pred["p05_hbm"]

    # -------------------------
    # Métricas del fold
    # -------------------------
    met = interval_metrics(
        y_true=test_pred["PM25_obs"].values,
        p05=test_pred["p05_hbm"].values,
        p50=test_pred["p50_hbm"].values,
        p95=test_pred["p95_hbm"].values,
        alpha=0.10
    )

    return idata, test_pred, met


print("CELDA 5 cargada correctamente.")
print("Función disponible: fit_hbm_m1_fold")

CELDA 5 cargada correctamente.
Función disponible: fit_hbm_m1_fold



 # CELDA 6 — Validación temporal del modelo base M1

 **Objetivo:**
 ejecutar la validación temporal del HBM base usando los 3 folds definidos.

 **Qué hace esta celda:**
 1. Recorre cada fold temporal.
 2. Escala las covariables usando únicamente los años de entrenamiento.
 3. Lleva las covariables escaladas al panel observado.
 4. Ajusta el modelo HBM base M1:
    - espacial ICAR
    - temporal RW1
 5. Predice sobre el período de prueba.
 6. Guarda:
    - predicciones del fold
    - resumen del posterior
    - parámetros de escalamiento
 7. Consolida:
    - métricas de todos los folds
    - predicciones conjuntas de prueba

 **Salidas principales:**
 - `HBM_PM25_M1_fold_1_pred_test.csv`
 - `HBM_PM25_M1_fold_2_pred_test.csv`
 - `HBM_PM25_M1_fold_3_pred_test.csv`
 - `HBM_PM25_M1_metrics_folds.csv`
 - `HBM_PM25_M1_pred_test_all_folds.csv`

 **Nota práctica:**
 esta es la primera corrida real del HBM. Puede tardar bastante
 dependiendo del equipo y del entorno de Python.

In [7]:
# %%
# =========================
# CELDA 6) Validación temporal M1
# =========================

all_metrics = []
all_test_preds = []

# puedes subir estos valores más adelante si quieres una corrida más exigente
DRAWS = 800
TUNE = 800
CHAINS = 4
TARGET_ACCEPT = 0.95
RANDOM_SEED = 42

for fold_name, fold_info in FOLDS.items():
    print("\n" + "=" * 80)
    print(f"Corriendo {fold_name}")
    print("Train years:", fold_info["train_years"])
    print("Test years :", fold_info["test_years"])

    # -------------------------------------------------
    # 1) Escalamiento usando solo años train
    # -------------------------------------------------
    grid_scaled, scale_params = scale_grid_by_train_years(
        grid_df=grid_base,
        x_cols=X_COLS_RAW,
        train_years=fold_info["train_years"]
    )

    # -------------------------------------------------
    # 2) Llevar covariables escaladas al panel observado
    # -------------------------------------------------
    obs_scaled = merge_scaled_covariates_to_obs(
        obs_df=obs_panel,
        grid_scaled=grid_scaled,
        x_cols=X_COLS_RAW
    )

    # -------------------------------------------------
    # 3) Ajustar modelo del fold
    # -------------------------------------------------
    idata_fold, test_pred_fold, met_fold = fit_hbm_m1_fold(
        obs_scaled=obs_scaled,
        grid_scaled=grid_scaled,
        w_edges_df=w_edges,
        x_cols_raw=X_COLS_RAW,
        train_years=fold_info["train_years"],
        test_years=fold_info["test_years"],
        n_cells=len(cell_ids),
        n_times=len(time_ids),
        draws=DRAWS,
        tune=TUNE,
        chains=CHAINS,
        target_accept=TARGET_ACCEPT,
        random_seed=RANDOM_SEED,
    )

    # -------------------------------------------------
    # 4) Guardar predicciones del fold
    # -------------------------------------------------
    test_pred_fold["fold"] = fold_name
    pred_fold_path = OUT_DIR / f"HBM_PM25_M1_{fold_name}_pred_test.csv"
    test_pred_fold.to_csv(pred_fold_path, index=False)

    # -------------------------------------------------
    # 5) Guardar resumen del posterior
    # -------------------------------------------------
    summary_fold = az.summary(
        idata_fold,
        var_names=["alpha", "beta", "sigma_y", "tau_phi", "sigma_t"],
        round_to=4
    )
    summary_fold_path = OUT_DIR / f"HBM_PM25_M1_{fold_name}_summary.csv"
    summary_fold.to_csv(summary_fold_path)

    # -------------------------------------------------
    # 6) Guardar parámetros de escalamiento
    # -------------------------------------------------
    scale_fold_path = OUT_DIR / f"HBM_PM25_M1_{fold_name}_scale_params.json"
    with open(scale_fold_path, "w", encoding="utf-8") as f:
        json.dump(scale_params, f, ensure_ascii=False, indent=2)

    # -------------------------------------------------
    # 7) Consolidar métricas
    # -------------------------------------------------
    met_fold["fold"] = fold_name
    met_fold["train_years"] = ",".join(map(str, fold_info["train_years"]))
    met_fold["test_years"] = ",".join(map(str, fold_info["test_years"]))
    all_metrics.append(met_fold)
    all_test_preds.append(test_pred_fold)

    print("\nGuardado del fold:")
    print("-", pred_fold_path)
    print("-", summary_fold_path)
    print("-", scale_fold_path)

    print("\nMétricas del fold:")
    for k, v in met_fold.items():
        print(f"- {k}: {v}")

# -------------------------------------------------
# 8) Consolidar todo
# -------------------------------------------------
metrics_df = pd.DataFrame(all_metrics)
preds_df = pd.concat(all_test_preds, ignore_index=True)

metrics_path = OUT_DIR / "HBM_PM25_M1_metrics_folds.csv"
preds_path = OUT_DIR / "HBM_PM25_M1_pred_test_all_folds.csv"

metrics_df.to_csv(metrics_path, index=False)
preds_df.to_csv(preds_path, index=False)

print("\n" + "=" * 80)
print("VALIDACIÓN TEMPORAL COMPLETADA")
print("- métricas consolidadas :", metrics_path)
print("- predicciones consolidadas :", preds_path)

print("\nResumen final de métricas:")
display(metrics_df)


Corriendo fold_1
Train years: [2020, 2021]
Test years : [2022]


Initializing NUTS using jitter+adapt_diag...
ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: local_subtensor_merge
ERROR (pytensor.graph.rewriting.basic): node: Subtensor{i}(Subtensor{start:}.0, 0)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 1920, in process_node
    replacements = node_rewriter.transform(
                   ^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 993, in transform
    return self.fn(fgraph, node)
           ^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 416, in local_subtensor_merge
    merge_two_slices(
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 

Output()

Sampling 4 chains for 800 tune and 800 draw iterations (3_200 + 3_200 draws total) took 85 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



Guardado del fold:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1_fold_1_pred_test.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1_fold_1_summary.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1_fold_1_scale_params.json

Métricas del fold:
- n: 165
- mae: 4.802806801668817
- rmse: 6.638664688351929
- bias: 3.6375110229670815
- r: 0.6636228611064005
- r2: 0.44039530178304487
- coverage_90: 1.0
- width_90_mean: 52.07300796233943
- wis_90: 52.07300796233943
- fold: fold_1
- train_years: 2020,2021
- test_years: 2022

Corriendo fold_2
Train years: [2020, 2021, 2022]
Test years : [2023]


...

Initializing NUTS using jitter+adapt_diag...
ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: local_subtensor_merge
ERROR (pytensor.graph.rewriting.basic): node: Subtensor{i}(Subtensor{start:}.0, 0)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 1920, in process_node
    replacements = node_rewriter.transform(
                   ^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 993, in transform
    return self.fn(fgraph, node)
           ^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 416, in local_subtensor_merge
    merge_two_slices(
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 

Output()

Sampling 4 chains for 800 tune and 800 draw iterations (3_200 + 3_200 draws total) took 89 seconds.
Chain 2 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



Guardado del fold:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1_fold_3_pred_test.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1_fold_3_summary.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1_fold_3_scale_params.json

Métricas del fold:
- n: 165
- mae: 6.034280602974204
- rmse: 7.320520937254306
- bias: 1.0552566770919647
- r: 0.5155130024080042
- r2: 0.265753655651715
- coverage_90: 0.8181818181818182
- width_90_mean: 34.18738170099168
- wis_90: 40.70457341699946
- fold: fold_3
- train_years: 2020,2021,2022,2023
- test_years: 2024

VALIDACIÓN TEMPORAL COMPLETADA
- métricas consolidadas : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1_metrics_folds.csv
- predicciones consolidadas : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1_pred_test_all_folds.csv

Resumen final de métricas:


,n,mae,rmse,bias,r,r2,coverage_90,width_90_mean,wis_90,fold,train_years,test_years
0,165,4.802807,6.638665,3.637511,0.663623,0.440395,1.000000,52.073008,52.073008,fold_1,"2020,2021",2022
1,180,3.298784,4.158264,1.288945,0.784301,0.615128,0.983333,40.227831,40.464161,fold_2,"2020,2021,2022",2023
2,165,6.034281,7.320521,1.055257,0.515513,0.265754,0.818182,34.187382,40.704573,fold_3,"2020,2021,2022,2023",2024



 # CELDA 6B — Redefinir el modelo base en versión más estable (M1b)

 **Objetivo:**
 crear una versión más robusta del HBM base para mejorar la convergencia
 sin cambiar la estructura sustantiva del modelo.

 **Qué cambia respecto al M1 inicial:**
 1. Se mantienen:
    - respuesta observada `PM25_obs`
    - efecto espacial ICAR
    - efecto temporal RW1
    - covariables A–D + viento
 2. Se ajustan priors para hacer el modelo más regularizado.
 3. Se mejora el muestreo con:
    - `target_accept` más alto
    - `max_treedepth` más alto
    - `init="adapt_diag"`

 **Objetivo metodológico:**
 reducir problemas de:
 - Rhat alto
 - ESS bajo
 - tree depth máximo

 **Importante:**
 esta celda solo redefine la función.
 En la siguiente la volvemos a correr por folds.

In [7]:
# %%
# =========================
# CELDA 6B) Versión estable del modelo base
# =========================

def fit_hbm_m1b_fold(
    obs_scaled,
    grid_scaled,
    w_edges_df,
    x_cols_raw,
    train_years,
    test_years,
    n_cells,
    n_times,
    draws=1000,
    tune=1500,
    chains=4,
    target_accept=0.99,
    max_treedepth=15,
    random_seed=42,
):
    """
    Ajusta el HBM base M1b para un fold temporal:
    - espacial ICAR
    - temporal RW1
    - respuesta observada PM25_obs
    - priors más regularizantes
    - sampler más conservador
    """

    z_cols = [f"{c}_z" for c in x_cols_raw]

    train_mask_obs = obs_scaled["year"].isin(train_years)
    test_mask_obs = obs_scaled["year"].isin(test_years)

    train_obs = obs_scaled.loc[train_mask_obs].copy()
    test_obs = obs_scaled.loc[test_mask_obs].copy()

    if len(train_obs) == 0:
        raise ValueError("No hay observaciones de entrenamiento para este fold.")
    if len(test_obs) == 0:
        raise ValueError("No hay observaciones de prueba para este fold.")

    # -------------------------
    # Datos train
    # -------------------------
    X_train = train_obs[z_cols].to_numpy(dtype=float)
    y_train_raw = train_obs["PM25_obs"].to_numpy(dtype=float)
    y_train = np.log(y_train_raw + EPS) if USE_LOG else y_train_raw

    cell_train = train_obs["cell_idx"].to_numpy(dtype=int)
    time_train = train_obs["time_idx"].to_numpy(dtype=int)

    # -------------------------
    # Datos test
    # -------------------------
    X_test = test_obs[z_cols].to_numpy(dtype=float)
    y_test = test_obs["PM25_obs"].to_numpy(dtype=float)
    cell_test = test_obs["cell_idx"].to_numpy(dtype=int)
    time_test = test_obs["time_idx"].to_numpy(dtype=int)

    # -------------------------
    # Vecindad espacial
    # -------------------------
    ei = w_edges_df["i"].to_numpy(dtype=int)
    ej = w_edges_df["j"].to_numpy(dtype=int)

    p = X_train.shape[1]
    alpha_mu = float(np.mean(y_train))

    with pm.Model() as model:
        # -------------------------
        # Efectos fijos más regularizados
        # -------------------------
        alpha = pm.Normal("alpha", mu=alpha_mu, sigma=1.0)
        beta = pm.Normal("beta", mu=0.0, sigma=0.5, shape=p)

        # -------------------------
        # Error observacional
        # -------------------------
        sigma_y = pm.HalfNormal("sigma_y", sigma=0.75)

        # -------------------------
        # Efecto espacial ICAR
        # -------------------------
        tau_phi = pm.Exponential("tau_phi", 2.0)
        phi_raw = pm.Normal("phi_raw", mu=0.0, sigma=1.0, shape=n_cells)
        phi = pm.Deterministic("phi", phi_raw - pt.mean(phi_raw))

        pm.Potential(
            "icar_penalty",
            -0.5 * tau_phi * pt.sum((phi[ei] - phi[ej]) ** 2)
        )

        # -------------------------
        # Efecto temporal RW1 más controlado
        # -------------------------
        sigma_t = pm.HalfNormal("sigma_t", sigma=0.25)
        delta_raw = pm.GaussianRandomWalk("delta_raw", sigma=sigma_t, shape=n_times)
        delta = pm.Deterministic("delta", delta_raw - pt.mean(delta_raw))

        # -------------------------
        # Media del modelo
        # -------------------------
        mu_train = alpha + pt.dot(X_train, beta) + phi[cell_train] + delta[time_train]

        # -------------------------
        # Likelihood
        # -------------------------
        pm.Normal("y_obs", mu=mu_train, sigma=sigma_y, observed=y_train)

        # -------------------------
        # Muestreo
        # -------------------------
        idata = pm.sample(
            draws=draws,
            tune=tune,
            chains=chains,
            init="adapt_diag",
            target_accept=target_accept,
            max_treedepth=max_treedepth,
            random_seed=random_seed,
            return_inferencedata=True,
            progressbar=True,
        )

    # -------------------------
    # Predicción test en escala original
    # -------------------------
    p05_test, p50_test, p95_test = posterior_predict_concentration(
        idata=idata,
        X_mat=X_test,
        cell_idx_arr=cell_test,
        time_idx_arr=time_test,
        eps=EPS
    )

    test_pred = test_obs[
        ["Estacion", "fecha", "year", "month", "cell_id", "cell_idx", "time_idx", "PM25_obs"]
    ].copy()

    test_pred["p05_hbm"] = p05_test
    test_pred["p50_hbm"] = p50_test
    test_pred["p95_hbm"] = p95_test
    test_pred["width_90"] = test_pred["p95_hbm"] - test_pred["p05_hbm"]

    met = interval_metrics(
        y_true=test_pred["PM25_obs"].values,
        p05=test_pred["p05_hbm"].values,
        p50=test_pred["p50_hbm"].values,
        p95=test_pred["p95_hbm"].values,
        alpha=0.10
    )

    return idata, test_pred, met


print("CELDA 6B cargada correctamente.")
print("Función disponible: fit_hbm_m1b_fold")

CELDA 6B cargada correctamente.
Función disponible: fit_hbm_m1b_fold


# CELDA 6C — Validación temporal del modelo estable M1b

 **Objetivo:**
 volver a ejecutar la validación temporal del HBM usando la versión
 más estable del modelo base:

 - espacial ICAR
 - temporal RW1
 - priors más regularizantes
 - muestreo más conservador

 **Qué hace esta celda:**
 1. Recorre los 3 folds temporales.
 2. Escala covariables usando solo train.
 3. Ajusta `fit_hbm_m1b_fold`.
 4. Guarda:
    - predicciones por fold
   - resumen posterior por fold
    - parámetros de escalamiento
 5. Consolida:
    - métricas de todos los folds
    - predicciones de prueba conjuntas
 6. Si existe el archivo de métricas del modelo M1 anterior,
    genera una tabla comparativa M1 vs M1b.

 **Salidas principales:**
 - `HBM_PM25_M1b_fold_1_pred_test.csv`
 - `HBM_PM25_M1b_fold_2_pred_test.csv`
 - `HBM_PM25_M1b_fold_3_pred_test.csv`
 - `HBM_PM25_M1b_metrics_folds.csv`
 - `HBM_PM25_M1b_pred_test_all_folds.csv`
 - `HBM_PM25_compare_M1_vs_M1b.csv` (si existe M1)

 **Qué esperamos observar:**
 - menor problema de convergencia
 - Rhat más cercano a 1
 - ESS más alto
 - calibración de intervalos más estable

In [9]:
# %%
# =========================
# CELDA 6C) Validación temporal M1b
# =========================

all_metrics_m1b = []
all_test_preds_m1b = []

# configuración más conservadora
DRAWS_B = 1000
TUNE_B = 1500
CHAINS_B = 4
TARGET_ACCEPT_B = 0.99
MAX_TREEDEPTH_B = 15
RANDOM_SEED_B = 42

for fold_name, fold_info in FOLDS.items():
    print("\n" + "=" * 80)
    print(f"Corriendo M1b - {fold_name}")
    print("Train years:", fold_info["train_years"])
    print("Test years :", fold_info["test_years"])

    # -------------------------------------------------
    # 1) Escalamiento usando solo train
    # -------------------------------------------------
    grid_scaled_b, scale_params_b = scale_grid_by_train_years(
        grid_df=grid_base,
        x_cols=X_COLS_RAW,
        train_years=fold_info["train_years"]
    )

    # -------------------------------------------------
    # 2) Llevar covariables escaladas al panel observado
    # -------------------------------------------------
    obs_scaled_b = merge_scaled_covariates_to_obs(
        obs_df=obs_panel,
        grid_scaled=grid_scaled_b,
        x_cols=X_COLS_RAW
    )

    # -------------------------------------------------
    # 3) Ajustar modelo del fold
    # -------------------------------------------------
    idata_fold_b, test_pred_fold_b, met_fold_b = fit_hbm_m1b_fold(
        obs_scaled=obs_scaled_b,
        grid_scaled=grid_scaled_b,
        w_edges_df=w_edges,
        x_cols_raw=X_COLS_RAW,
        train_years=fold_info["train_years"],
        test_years=fold_info["test_years"],
        n_cells=len(cell_ids),
        n_times=len(time_ids),
        draws=DRAWS_B,
        tune=TUNE_B,
        chains=CHAINS_B,
        target_accept=TARGET_ACCEPT_B,
        max_treedepth=MAX_TREEDEPTH_B,
        random_seed=RANDOM_SEED_B,
    )

    # -------------------------------------------------
    # 4) Guardar predicciones del fold
    # -------------------------------------------------
    test_pred_fold_b["fold"] = fold_name
    pred_fold_path_b = OUT_DIR / f"HBM_PM25_M1b_{fold_name}_pred_test.csv"
    test_pred_fold_b.to_csv(pred_fold_path_b, index=False)

    # -------------------------------------------------
    # 5) Guardar resumen del posterior
    # -------------------------------------------------
    summary_fold_b = az.summary(
        idata_fold_b,
        var_names=["alpha", "beta", "sigma_y", "tau_phi", "sigma_t"],
        round_to=4
    )
    summary_fold_path_b = OUT_DIR / f"HBM_PM25_M1b_{fold_name}_summary.csv"
    summary_fold_b.to_csv(summary_fold_path_b)

    # -------------------------------------------------
    # 6) Guardar parámetros de escalamiento
    # -------------------------------------------------
    scale_fold_path_b = OUT_DIR / f"HBM_PM25_M1b_{fold_name}_scale_params.json"
    with open(scale_fold_path_b, "w", encoding="utf-8") as f:
        json.dump(scale_params_b, f, ensure_ascii=False, indent=2)

    # -------------------------------------------------
    # 7) Consolidar métricas
    # -------------------------------------------------
    met_fold_b["model"] = "M1b"
    met_fold_b["fold"] = fold_name
    met_fold_b["train_years"] = ",".join(map(str, fold_info["train_years"]))
    met_fold_b["test_years"] = ",".join(map(str, fold_info["test_years"]))

    all_metrics_m1b.append(met_fold_b)
    all_test_preds_m1b.append(test_pred_fold_b)

    print("\nGuardado del fold:")
    print("-", pred_fold_path_b)
    print("-", summary_fold_path_b)
    print("-", scale_fold_path_b)

    print("\nMétricas del fold:")
    for k, v in met_fold_b.items():
        print(f"- {k}: {v}")

# -------------------------------------------------
# 8) Consolidar resultados M1b
# -------------------------------------------------
metrics_m1b_df = pd.DataFrame(all_metrics_m1b)
preds_m1b_df = pd.concat(all_test_preds_m1b, ignore_index=True)

metrics_m1b_path = OUT_DIR / "HBM_PM25_M1b_metrics_folds.csv"
preds_m1b_path = OUT_DIR / "HBM_PM25_M1b_pred_test_all_folds.csv"

metrics_m1b_df.to_csv(metrics_m1b_path, index=False)
preds_m1b_df.to_csv(preds_m1b_path, index=False)

print("\n" + "=" * 80)
print("VALIDACIÓN TEMPORAL M1b COMPLETADA")
print("- métricas consolidadas :", metrics_m1b_path)
print("- predicciones consolidadas :", preds_m1b_path)

print("\nResumen final de métricas M1b:")
display(metrics_m1b_df)

# -------------------------------------------------
# 9) Comparar M1 vs M1b si existe archivo anterior
# -------------------------------------------------
m1_metrics_path = OUT_DIR / "HBM_PM25_M1_metrics_folds.csv"

if m1_metrics_path.exists():
    metrics_m1_df = pd.read_csv(m1_metrics_path).copy()
    metrics_m1_df["model"] = "M1"

    cols_keep = [
        "model", "fold", "n", "mae", "rmse", "bias", "r", "r2",
        "coverage_90", "width_90_mean", "wis_90", "train_years", "test_years"
    ]

    compare_df = pd.concat(
        [
            metrics_m1_df[cols_keep],
            metrics_m1b_df[cols_keep]
        ],
        ignore_index=True
    )

    compare_path = OUT_DIR / "HBM_PM25_compare_M1_vs_M1b.csv"
    compare_df.to_csv(compare_path, index=False)

    print("\nComparación M1 vs M1b guardada en:")
    print("-", compare_path)

    print("\nTabla comparativa:")
    display(compare_df.sort_values(["fold", "model"]).reset_index(drop=True))
else:
    print("\nNo se encontró el archivo de métricas de M1 previo.")
    print("Se omitió la comparación M1 vs M1b.")


Corriendo M1b - fold_1
Train years: [2020, 2021]
Test years : [2022]


Initializing NUTS using adapt_diag...
ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: local_subtensor_merge
ERROR (pytensor.graph.rewriting.basic): node: Subtensor{i}(Subtensor{start:}.0, 0)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 1920, in process_node
    replacements = node_rewriter.transform(
                   ^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 993, in transform
    return self.fn(fgraph, node)
           ^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 416, in local_subtensor_merge
    merge_two_slices(
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 865, in

Output()

Sampling 4 chains for 1_500 tune and 1_000 draw iterations (6_000 + 4_000 draws total) took 2287 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



Guardado del fold:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1b_fold_1_pred_test.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1b_fold_1_summary.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1b_fold_1_scale_params.json

Métricas del fold:
- n: 165
- mae: 5.13171469396694
- rmse: 7.064043920507481
- bias: 4.057671145871742
- r: 0.6509028772254284
- r2: 0.4236745555803411
- coverage_90: 1.0
- width_90_mean: 48.205675361006655
- wis_90: 48.205675361006655
- model: M1b
- fold: fold_1
- train_years: 2020,2021
- test_years: 2022

Corriendo M1b - fold_2
Train years: [2020, 2021, 2022]
Test years : [2023]


...

Sampling 4 chains for 1_500 tune and 1_000 draw iterations (6_000 + 4_000 draws total) took 2349 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



Guardado del fold:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1b_fold_3_pred_test.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1b_fold_3_summary.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1b_fold_3_scale_params.json

Métricas del fold:
- n: 165
- mae: 5.729405102009341
- rmse: 7.120183518602233
- bias: 0.420366672646799
- r: 0.5326443358859322
- r2: 0.28370998855136575
- coverage_90: 0.8181818181818182
- width_90_mean: 33.703959242328445
- wis_90: 40.21787883838249
- model: M1b
- fold: fold_3
- train_years: 2020,2021,2022,2023
- test_years: 2024

VALIDACIÓN TEMPORAL M1b COMPLETADA
- métricas consolidadas : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1b_metrics_folds.csv
- predicciones consolidadas : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1b_pred_test_all_folds.csv

Resumen final de métricas M1b:


,n,mae,rmse,bias,r,r2,coverage_90,width_90_mean,wis_90,model,fold,train_years,test_years
0,165,5.131715,7.064044,4.057671,0.650903,0.423675,1.000000,48.205675,48.205675,M1b,fold_1,"2020,2021",2022
1,180,3.331843,4.197950,1.411280,0.784936,0.616125,0.983333,40.585155,40.879425,M1b,fold_2,"2020,2021,2022",2023
2,165,5.729405,7.120184,0.420367,0.532644,0.283710,0.818182,33.703959,40.217879,M1b,fold_3,"2020,2021,2022,2023",2024



Comparación M1 vs M1b guardada en:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_compare_M1_vs_M1b.csv

Tabla comparativa:


,model,fold,n,mae,rmse,bias,r,r2,coverage_90,width_90_mean,wis_90,train_years,test_years
0,M1,fold_1,165,4.802807,6.638665,3.637511,0.663623,0.440395,1.000000,52.073008,52.073008,"2020,2021",2022
1,M1b,fold_1,165,5.131715,7.064044,4.057671,0.650903,0.423675,1.000000,48.205675,48.205675,"2020,2021",2022
2,M1,fold_2,180,3.298784,4.158264,1.288945,0.784301,0.615128,0.983333,40.227831,40.464161,"2020,2021,2022",2023
3,M1b,fold_2,180,3.331843,4.197950,1.411280,0.784936,0.616125,0.983333,40.585155,40.879425,"2020,2021,2022",2023
4,M1,fold_3,165,6.034281,7.320521,1.055257,0.515513,0.265754,0.818182,34.187382,40.704573,"2020,2021,2022,2023",2024
5,M1b,fold_3,165,5.729405,7.120184,0.420367,0.532644,0.283710,0.818182,33.703959,40.217879,"2020,2021,2022,2023",2024



# CELDA 6D — Comparación de diagnósticos de convergencia: M1 vs M1b

 **Objetivo:**
 comparar formalmente la calidad del muestreo bayesiano entre los modelos
 M1 y M1b usando los archivos `summary.csv` guardados por fold.

 **Qué hace esta celda:**
 1. Lee los archivos resumen de:
    - `HBM_PM25_M1_fold_*_summary.csv`
    - `HBM_PM25_M1b_fold_*_summary.csv`
 2. Extrae, para cada fold:
    - máximo `r_hat`
    - mínimo `ess_bulk`
    - mínimo `ess_tail`
 3. Consolida la comparación M1 vs M1b.
 4. Marca reglas simples de interpretación:
    - `r_hat_ok`: max r_hat <= 1.01
    - `ess_bulk_ok`: min ess_bulk >= 400
    - `ess_tail_ok`: min ess_tail >= 400

 **Interpretación esperada:**
 - un buen modelo debe tener `r_hat` cercano a 1
 - ESS no debería ser muy bajo
 - si M1b mejora claramente estos indicadores, será preferible a M1
   aunque las métricas predictivas sean parecidas

In [10]:
# %%
# =========================
# CELDA 6D) Diagnósticos M1 vs M1b
# =========================

from pathlib import Path

summary_files = {
    "M1": {
        "fold_1": OUT_DIR / "HBM_PM25_M1_fold_1_summary.csv",
        "fold_2": OUT_DIR / "HBM_PM25_M1_fold_2_summary.csv",
        "fold_3": OUT_DIR / "HBM_PM25_M1_fold_3_summary.csv",
    },
    "M1b": {
        "fold_1": OUT_DIR / "HBM_PM25_M1b_fold_1_summary.csv",
        "fold_2": OUT_DIR / "HBM_PM25_M1b_fold_2_summary.csv",
        "fold_3": OUT_DIR / "HBM_PM25_M1b_fold_3_summary.csv",
    }
}

rows = []

for model_name, model_files in summary_files.items():
    for fold_name, path_summary in model_files.items():
        if not path_summary.exists():
            print(f"No existe: {path_summary}")
            continue

        df_sum = pd.read_csv(path_summary, index_col=0)

        # por seguridad, revisar que las columnas existan
        needed_cols = ["r_hat", "ess_bulk", "ess_tail"]
        for c in needed_cols:
            if c not in df_sum.columns:
                raise ValueError(f"Falta la columna '{c}' en {path_summary.name}")

        row = {
            "model": model_name,
            "fold": fold_name,
            "max_r_hat": df_sum["r_hat"].max(),
            "min_ess_bulk": df_sum["ess_bulk"].min(),
            "min_ess_tail": df_sum["ess_tail"].min(),
        }

        row["r_hat_ok"] = row["max_r_hat"] <= 1.01
        row["ess_bulk_ok"] = row["min_ess_bulk"] >= 400
        row["ess_tail_ok"] = row["min_ess_tail"] >= 400

        rows.append(row)

diag_compare = pd.DataFrame(rows).sort_values(["fold", "model"]).reset_index(drop=True)

diag_compare_path = OUT_DIR / "HBM_PM25_compare_diagnostics_M1_vs_M1b.csv"
diag_compare.to_csv(diag_compare_path, index=False)

print("Diagnósticos comparativos guardados en:")
print("-", diag_compare_path)

print("\nTabla comparativa de convergencia:")
display(diag_compare)

Diagnósticos comparativos guardados en:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_compare_diagnostics_M1_vs_M1b.csv

Tabla comparativa de convergencia:


,model,fold,max_r_hat,min_ess_bulk,min_ess_tail,r_hat_ok,ess_bulk_ok,ess_tail_ok
0,M1,fold_1,1.0087,785.7483,1504.4709,True,True,True
1,M1b,fold_1,1.0017,1091.8871,1808.5574,True,True,True
2,M1,fold_2,1.0101,777.6515,1536.7774,False,True,True
3,M1b,fold_2,1.0043,874.8228,1758.3279,True,True,True
4,M1,fold_3,1.0134,133.6106,315.1460,False,False,False
5,M1b,fold_3,1.0021,429.2079,899.2738,True,True,True


# # CELDA 7B — Ajuste final del modelo seleccionado M1b y exportación de la superficie

 **Objetivo:**
 ajustar el modelo final seleccionado (**M1b**) usando todas las observaciones
 válidas de PM2.5 entre 2020 y 2024, para producir la superficie final celda–mes.

 **Modelo final seleccionado:**
 - Respuesta observada: `PM25_obs`
 - Efecto espacial: `ICAR`
 - Efecto temporal: `RW1`
 - Priors más regularizantes
 - Muestreo más estable

 **Qué hace esta celda:**
 1. Escala covariables usando todo el período 2020–2024.
 2. Lleva las covariables escaladas al panel observado.
 3. Ajusta el modelo final M1b con todas las observaciones válidas.
 4. Predice sobre toda la malla 3 km y todos los meses.
 5. Exporta:
    - `p05_hbm`
    - `p50_hbm`
    - `p95_hbm`
    - `width_90`
 6. Guarda también:
    - resumen del posterior
    - diagnósticos finales
    - parámetros de escalamiento
 **Salidas principales:**
 - `HBM_PM25_M1b_surface_final.csv`
 - `HBM_PM25_M1b_final_summary.csv`
 - `HBM_PM25_M1b_final_diagnostics.csv`
 - `HBM_PM25_M1b_final_scale_params.json`

 **Nota:**
 esta corrida puede tardar más que los folds, porque usa todo el período completo.

In [11]:
# %%
# =========================
# CELDA 7B) Ajuste final M1b
# =========================

ALL_YEARS = [2020, 2021, 2022, 2023, 2024]

DRAWS_FINAL = 1200
TUNE_FINAL = 1800
CHAINS_FINAL = 4
TARGET_ACCEPT_FINAL = 0.99
MAX_TREEDEPTH_FINAL = 15
RANDOM_SEED_FINAL = 42

# -------------------------------------------------
# 1) Escalar covariables con todo el período
# -------------------------------------------------
grid_scaled_all, scale_params_all = scale_grid_by_train_years(
    grid_df=grid_base,
    x_cols=X_COLS_RAW,
    train_years=ALL_YEARS
)

obs_scaled_all = merge_scaled_covariates_to_obs(
    obs_df=obs_panel,
    grid_scaled=grid_scaled_all,
    x_cols=X_COLS_RAW
)

z_cols = [f"{c}_z" for c in X_COLS_RAW]

# -------------------------------------------------
# 2) Arrays de entrenamiento completo
# -------------------------------------------------
X_all = obs_scaled_all[z_cols].to_numpy(dtype=float)
y_all_raw = obs_scaled_all["PM25_obs"].to_numpy(dtype=float)
y_all = np.log(y_all_raw + EPS) if USE_LOG else y_all_raw

cell_all = obs_scaled_all["cell_idx"].to_numpy(dtype=int)
time_all = obs_scaled_all["time_idx"].to_numpy(dtype=int)

ei = w_edges["i"].to_numpy(dtype=int)
ej = w_edges["j"].to_numpy(dtype=int)

p = X_all.shape[1]
alpha_mu_all = float(np.mean(y_all))

print("Ajustando modelo final M1b...")
print("- observaciones válidas:", len(obs_scaled_all))
print("- estaciones únicas    :", obs_scaled_all["Estacion"].nunique())
print("- celdas observadas     :", obs_scaled_all["cell_id"].nunique())
print("- celdas totales malla  :", len(cell_ids))
print("- meses totales         :", len(time_ids))

# -------------------------------------------------
# 3) Ajuste final M1b
# -------------------------------------------------
with pm.Model() as final_model_m1b:
    # efectos fijos regularizados
    alpha = pm.Normal("alpha", mu=alpha_mu_all, sigma=1.0)
    beta = pm.Normal("beta", mu=0.0, sigma=0.5, shape=p)

    # error observacional
    sigma_y = pm.HalfNormal("sigma_y", sigma=0.75)

    # espacial ICAR
    tau_phi = pm.Exponential("tau_phi", 2.0)
    phi_raw = pm.Normal("phi_raw", mu=0.0, sigma=1.0, shape=len(cell_ids))
    phi = pm.Deterministic("phi", phi_raw - pt.mean(phi_raw))
    pm.Potential("icar_penalty", -0.5 * tau_phi * pt.sum((phi[ei] - phi[ej]) ** 2))

    # temporal RW1
    sigma_t = pm.HalfNormal("sigma_t", sigma=0.25)
    delta_raw = pm.GaussianRandomWalk("delta_raw", sigma=sigma_t, shape=len(time_ids))
    delta = pm.Deterministic("delta", delta_raw - pt.mean(delta_raw))

    # media
    mu_all = alpha + pt.dot(X_all, beta) + phi[cell_all] + delta[time_all]

    # likelihood
    pm.Normal("y_obs", mu=mu_all, sigma=sigma_y, observed=y_all)

    # muestreo
    idata_final_m1b = pm.sample(
        draws=DRAWS_FINAL,
        tune=TUNE_FINAL,
        chains=CHAINS_FINAL,
        init="adapt_diag",
        target_accept=TARGET_ACCEPT_FINAL,
        max_treedepth=MAX_TREEDEPTH_FINAL,
        random_seed=RANDOM_SEED_FINAL,
        return_inferencedata=True,
        progressbar=True,
    )

# -------------------------------------------------
# 4) Predicción sobre toda la malla y todos los meses
# -------------------------------------------------
X_grid_all = grid_scaled_all[z_cols].to_numpy(dtype=float)
cell_grid_all = grid_scaled_all["cell_idx"].to_numpy(dtype=int)
time_grid_all = grid_scaled_all["time_idx"].to_numpy(dtype=int)

p05_grid, p50_grid, p95_grid = posterior_predict_concentration(
    idata=idata_final_m1b,
    X_mat=X_grid_all,
    cell_idx_arr=cell_grid_all,
    time_idx_arr=time_grid_all,
    eps=EPS
)

surface_final_m1b = grid_scaled_all[["cell_id", "fecha", "year", "month", "cell_idx", "time_idx"]].copy()
surface_final_m1b["p05_hbm"] = p05_grid
surface_final_m1b["p50_hbm"] = p50_grid
surface_final_m1b["p95_hbm"] = p95_grid
surface_final_m1b["width_90"] = surface_final_m1b["p95_hbm"] - surface_final_m1b["p05_hbm"]

# -------------------------------------------------
# 5) Resumen y diagnósticos finales
# -------------------------------------------------
summary_final_m1b = az.summary(
    idata_final_m1b,
    var_names=["alpha", "beta", "sigma_y", "tau_phi", "sigma_t"],
    round_to=4
)

diag_final_m1b = pd.DataFrame([{
    "model": "M1b_final",
    "max_r_hat": float(summary_final_m1b["r_hat"].max()),
    "min_ess_bulk": float(summary_final_m1b["ess_bulk"].min()),
    "min_ess_tail": float(summary_final_m1b["ess_tail"].min()),
    "r_hat_ok": bool(summary_final_m1b["r_hat"].max() <= 1.01),
    "ess_bulk_ok": bool(summary_final_m1b["ess_bulk"].min() >= 400),
    "ess_tail_ok": bool(summary_final_m1b["ess_tail"].min() >= 400),
}])

# -------------------------------------------------
# 6) Guardar salidas
# -------------------------------------------------
surface_final_path = OUT_DIR / "HBM_PM25_M1b_surface_final.csv"
summary_final_path = OUT_DIR / "HBM_PM25_M1b_final_summary.csv"
diag_final_path = OUT_DIR / "HBM_PM25_M1b_final_diagnostics.csv"
scale_final_path = OUT_DIR / "HBM_PM25_M1b_final_scale_params.json"
idata_final_path = OUT_DIR / "HBM_PM25_M1b_final_posterior.nc"

surface_final_m1b.to_csv(surface_final_path, index=False)
summary_final_m1b.to_csv(summary_final_path)
diag_final_m1b.to_csv(diag_final_path, index=False)

with open(scale_final_path, "w", encoding="utf-8") as f:
    json.dump(scale_params_all, f, ensure_ascii=False, indent=2)

az.to_netcdf(idata_final_m1b, idata_final_path)

# -------------------------------------------------
# 7) Resumen final
# -------------------------------------------------
print("\nAJUSTE FINAL M1b COMPLETADO")
print("- superficie final     :", surface_final_path)
print("- summary final        :", summary_final_path)
print("- diagnósticos finales :", diag_final_path)
print("- scale params         :", scale_final_path)
print("- posterior netcdf     :", idata_final_path)

print("\nDiagnósticos finales:")
display(diag_final_m1b)

print("\nPrimeras filas de la superficie final:")
display(surface_final_m1b.head())

Ajustando modelo final M1b...
- observaciones válidas: 808
- estaciones únicas    : 16
- celdas observadas     : 14
- celdas totales malla  : 254
- meses totales         : 60


Initializing NUTS using adapt_diag...
ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: local_subtensor_merge
ERROR (pytensor.graph.rewriting.basic): node: Subtensor{i}(Subtensor{start:}.0, 0)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 1920, in process_node
    replacements = node_rewriter.transform(
                   ^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 993, in transform
    return self.fn(fgraph, node)
           ^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 416, in local_subtensor_merge
    merge_two_slices(
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 865, in

Output()

Sampling 4 chains for 1_800 tune and 1_200 draw iterations (7_200 + 4_800 draws total) took 4891 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



AJUSTE FINAL M1b COMPLETADO
- superficie final     : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1b_surface_final.csv
- summary final        : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1b_final_summary.csv
- diagnósticos finales : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1b_final_diagnostics.csv
- scale params         : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1b_final_scale_params.json
- posterior netcdf     : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1b_final_posterior.nc

Diagnósticos finales:


,model,max_r_hat,min_ess_bulk,min_ess_tail,r_hat_ok,ess_bulk_ok,ess_tail_ok
0,M1b_final,1.008,270.0316,504.332,True,False,True



Primeras filas de la superficie final:


,cell_id,fecha,year,month,cell_idx,time_idx,p05_hbm,p50_hbm,p95_hbm,width_90
0,0,2020-01-01,2020,1,0,0,2.691060,15.194307,81.657159,78.966100
1,1,2020-01-01,2020,1,1,0,2.818001,15.060520,80.211588,77.393586
2,2,2020-01-01,2020,1,2,0,2.871660,14.941641,79.147653,76.275993
3,41,2020-01-01,2020,1,3,0,2.981552,15.387290,79.938046,76.956494
4,42,2020-01-01,2020,1,4,0,2.591232,15.403391,89.183165,86.591933



# CELDA 8 — Revisión de coherencia de la superficie final PM2.5

 **Objetivo:**
 hacer una validación descriptiva básica de la superficie final del HBM
 antes de usarla en mapas, análisis de sensibilidad o etapas posteriores.

**Qué hace esta celda:**
 1. Lee la superficie final `HBM_PM25_M1b_surface_final.csv`.
 2. Verifica dimensiones esperadas:
    - número de filas
    - número de celdas
    - número de meses
 3. Resume estadísticamente:
    - `p05_hbm`
    - `p50_hbm`
    - `p95_hbm`
    - `width_90`
 4. Revisa si hay:
    - nulos
    - valores negativos
    - intervalos inconsistentes
 5. Genera un resumen anual del percentil central `p50_hbm`.

 **Objetivo metodológico:**
 confirmar que la superficie final es coherente y lista para análisis posterior.

In [9]:
# %%
# =========================
# CELDA 8) Revisión de coherencia de la superficie final
# =========================

surface_path = OUT_DIR / "HBM_PM25_M1b_surface_final.csv"

surface = pd.read_csv(surface_path, parse_dates=["fecha"])

print("Archivo leído:", surface_path)
print("\nDimensiones generales:")
print("- filas totales   :", len(surface))
print("- celdas únicas   :", surface["cell_id"].nunique())
print("- meses únicos    :", surface["fecha"].nunique())
print("- años presentes  :", sorted(surface["year"].unique().tolist()))

print("\nResumen descriptivo:")
display(surface[["p05_hbm", "p50_hbm", "p95_hbm", "width_90"]].describe())

print("\nChequeos de consistencia:")
print("- nulos en p05_hbm    :", surface["p05_hbm"].isna().sum())
print("- nulos en p50_hbm    :", surface["p50_hbm"].isna().sum())
print("- nulos en p95_hbm    :", surface["p95_hbm"].isna().sum())
print("- nulos en width_90   :", surface["width_90"].isna().sum())

print("- p50 negativos       :", (surface["p50_hbm"] < 0).sum())
print("- width_90 negativos  :", (surface["width_90"] < 0).sum())
print("- casos p05 > p50     :", (surface["p05_hbm"] > surface["p50_hbm"]).sum())
print("- casos p50 > p95     :", (surface["p50_hbm"] > surface["p95_hbm"]).sum())

# resumen anual
annual_summary = (
    surface.groupby("year")["p50_hbm"]
    .agg(["count", "mean", "std", "min", "median", "max"])
    .reset_index()
)

annual_summary_path = OUT_DIR / "HBM_PM25_M1b_surface_annual_summary.csv"
annual_summary.to_csv(annual_summary_path, index=False)

print("\nResumen anual de p50_hbm:")
display(annual_summary)

# extremos para inspección
top_high = surface.nlargest(10, "p50_hbm")[["cell_id", "fecha", "year", "month", "p50_hbm", "p95_hbm", "width_90"]]
top_low = surface.nsmallest(10, "p50_hbm")[["cell_id", "fecha", "year", "month", "p50_hbm", "p05_hbm", "width_90"]]

print("\nTop 10 p50_hbm más altos:")
display(top_high)

print("\nTop 10 p50_hbm más bajos:")
display(top_low)

print("\nArchivo guardado:")
print("-", annual_summary_path)

Archivo leído: d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1b_surface_final.csv

Dimensiones generales:
- filas totales   : 15240
- celdas únicas   : 254
- meses únicos    : 60
- años presentes  : [2020, 2021, 2022, 2023, 2024]

Resumen descriptivo:


,p05_hbm,p50_hbm,p95_hbm,width_90
count,15240.000000,15240.000000,15240.000000,15240.000000
mean,3.305328,14.684733,76.643780,73.338452
std,3.020802,4.904145,29.484263,30.073289
min,1.084324,4.764005,5.263141,0.934124
25%,2.029218,10.914194,56.577118,54.723441
50%,2.762701,14.650753,77.023502,74.422535
75%,3.364164,17.769652,95.748457,92.495148
max,39.563858,43.761062,205.499358,199.214372



Chequeos de consistencia:
- nulos en p05_hbm    : 0
- nulos en p50_hbm    : 0
- nulos en p95_hbm    : 0
- nulos en width_90   : 0
- p50 negativos       : 0
- width_90 negativos  : 0
- casos p05 > p50     : 0
- casos p50 > p95     : 0

Resumen anual de p50_hbm:


,year,count,mean,std,min,median,max
0,2020,3048,14.722088,6.194751,6.021274,13.690033,43.761062
1,2021,3048,13.802466,3.841481,5.190119,13.752242,27.253179
2,2022,3048,15.636099,3.407832,7.021261,15.947074,31.431892
3,2023,3048,14.054604,4.387966,5.090253,14.635855,34.879715
4,2024,3048,15.208409,5.824804,4.764005,14.375973,41.850666



Top 10 p50_hbm más altos:


,cell_id,fecha,year,month,p50_hbm,p95_hbm,width_90
644,442,2020-03-01,2020,3,43.761062,48.615102,9.051244
12862,482,2024-03-01,2024,3,41.850666,45.963516,7.815061
670,482,2020-03-01,2020,3,41.550149,45.990111,8.336508
673,485,2020-03-01,2020,3,41.142649,45.464686,8.203158
12865,485,2024-03-01,2024,3,39.019955,42.701707,7.231711
643,441,2020-03-01,2020,3,38.474502,43.867669,10.315733
12836,442,2024-03-01,2024,3,38.228616,42.219711,7.383794
671,483,2020-03-01,2020,3,35.812839,205.499358,199.214372
694,523,2020-03-01,2020,3,35.631394,184.919917,178.352652
12835,441,2024-03-01,2024,3,35.076398,39.894188,9.123718



Top 10 p50_hbm más bajos:


,cell_id,fecha,year,month,p50_hbm,p05_hbm,width_90
13940,611,2024-07-01,2024,7,4.764005,4.315921,0.947221
13952,651,2024-07-01,2024,7,4.837261,4.404619,0.934124
10638,611,2023-06-01,2023,6,5.090253,4.625744,1.007794
4796,611,2021-07-01,2021,7,5.190119,4.689888,1.050686
10650,651,2023-06-01,2023,6,5.267091,4.768071,1.066916
13920,564,2024-07-01,2024,7,5.379337,4.901605,1.004151
4808,651,2021-07-01,2021,7,5.469149,4.943641,1.097002
10892,611,2023-07-01,2023,7,5.507484,4.978048,1.100835
10904,651,2023-07-01,2023,7,5.711238,5.161706,1.148651
10618,564,2023-06-01,2023,6,5.717079,5.205631,1.077853



Archivo guardado:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1b_surface_annual_summary.csv



# CELDA 9A — Auditoría de calidad del panel observado PM2.5

 **Objetivo:**
 revisar la calidad del panel observado de estaciones antes de hacer limpieza
 o volver a ajustar el HBM.

 **Qué hace esta celda:**
 1. Lee el archivo corregido de estaciones.
 2. Construye la fecha mensual.
 3. Filtra observaciones válidas de PM2.5.
 4. Revisa duplicados por estación y mes.
 5. Resume la distribución de PM2.5 por estación:
    - número de observaciones
    - media
    - desviación estándar
    - percentiles
    - mínimo y máximo
 6. Marca observaciones atípicas por estación usando una regla robusta IQR.
 7. Genera una tabla con las filas sospechosas.
 8. Guarda archivos de auditoría para usarlos en la siguiente etapa de limpieza.

 **Importante:**
 esta celda no elimina datos.
 Solo identifica dónde puede estar entrando ruido al modelo.

In [10]:
# %%
# =========================
# CELDA 9A) Auditoría de calidad del panel observado PM2.5
# =========================

obs_audit = pd.read_csv(OBS_CSV)

# -------------------------------------------------
# 1) Fecha mensual
# -------------------------------------------------
obs_audit["fecha"] = pd.to_datetime(
    dict(year=obs_audit["Año"].astype(int), month=obs_audit["Mes"].astype(int), day=1),
    errors="coerce"
)

if obs_audit["fecha"].isna().any():
    raise ValueError("Hay fechas inválidas en el panel observado.")

# -------------------------------------------------
# 2) Filtrar PM2.5 válido
# -------------------------------------------------
obs_audit["valido_PM25"] = obs_audit["valido_PM25"].astype(bool)

pm_obs = obs_audit.loc[
    (obs_audit["valido_PM25"] == True) &
    (obs_audit["PM25"].notna())
].copy()

pm_obs = pm_obs.rename(columns={"PM25": "PM25_obs"})

print("Resumen general del panel observado PM2.5")
print("- filas totales en archivo            :", len(obs_audit))
print("- filas válidas PM2.5                :", len(pm_obs))
print("- estaciones con PM2.5 válido        :", pm_obs["Estacion"].nunique())
print("- meses observados PM2.5             :", pm_obs["fecha"].nunique())
print("- rango temporal                     :", pm_obs["fecha"].min().date(), "a", pm_obs["fecha"].max().date())

# -------------------------------------------------
# 3) Duplicados por estación-mes
# -------------------------------------------------
dup_mask = pm_obs.duplicated(subset=["Estacion", "fecha"], keep=False)
dup_rows = pm_obs.loc[dup_mask].sort_values(["Estacion", "fecha"]).copy()

print("\nDuplicados por estación-mes:")
print("- número de filas duplicadas:", len(dup_rows))
print("- número de combinaciones duplicadas:",
      dup_rows[["Estacion", "fecha"]].drop_duplicates().shape[0])

# -------------------------------------------------
# 4) Resumen por estación
# -------------------------------------------------
station_summary = (
    pm_obs.groupby("Estacion")["PM25_obs"]
    .agg(
        n="count",
        mean="mean",
        std="std",
        min="min",
        q01=lambda s: s.quantile(0.01),
        q05=lambda s: s.quantile(0.05),
        q25=lambda s: s.quantile(0.25),
        median="median",
        q75=lambda s: s.quantile(0.75),
        q95=lambda s: s.quantile(0.95),
        q99=lambda s: s.quantile(0.99),
        max="max",
    )
    .reset_index()
    .sort_values("mean", ascending=False)
)

# -------------------------------------------------
# 5) Regla robusta IQR por estación
# -------------------------------------------------
bounds = (
    pm_obs.groupby("Estacion")["PM25_obs"]
    .agg(
        q1=lambda s: s.quantile(0.25),
        q3=lambda s: s.quantile(0.75)
    )
    .reset_index()
)

bounds["iqr"] = bounds["q3"] - bounds["q1"]
bounds["lower_iqr15"] = bounds["q1"] - 1.5 * bounds["iqr"]
bounds["upper_iqr15"] = bounds["q3"] + 1.5 * bounds["iqr"]
bounds["lower_iqr30"] = bounds["q1"] - 3.0 * bounds["iqr"]
bounds["upper_iqr30"] = bounds["q3"] + 3.0 * bounds["iqr"]

pm_obs = pm_obs.merge(bounds, on="Estacion", how="left")

pm_obs["flag_iqr15"] = (
    (pm_obs["PM25_obs"] < pm_obs["lower_iqr15"]) |
    (pm_obs["PM25_obs"] > pm_obs["upper_iqr15"])
)

pm_obs["flag_iqr30"] = (
    (pm_obs["PM25_obs"] < pm_obs["lower_iqr30"]) |
    (pm_obs["PM25_obs"] > pm_obs["upper_iqr30"])
)

# -------------------------------------------------
# 6) Tabla de observaciones sospechosas
# -------------------------------------------------
flagged_rows = pm_obs.loc[
    pm_obs["flag_iqr15"] == True,
    [
        "Estacion", "fecha", "Año", "Mes", "PM25_obs",
        "Temp_media", "HR", "Vel_viento", "Presión", "Precipitación",
        "q1", "q3", "iqr", "lower_iqr15", "upper_iqr15", "flag_iqr15", "flag_iqr30"
    ]
].sort_values(["Estacion", "fecha"])

# -------------------------------------------------
# 7) Resumen de flags por estación
# -------------------------------------------------
flag_summary = (
    pm_obs.groupby("Estacion")
    .agg(
        n_total=("PM25_obs", "count"),
        n_flag_iqr15=("flag_iqr15", "sum"),
        n_flag_iqr30=("flag_iqr30", "sum"),
        pm25_mean=("PM25_obs", "mean"),
        pm25_std=("PM25_obs", "std"),
        pm25_min=("PM25_obs", "min"),
        pm25_max=("PM25_obs", "max")
    )
    .reset_index()
)

flag_summary["pct_flag_iqr15"] = 100 * flag_summary["n_flag_iqr15"] / flag_summary["n_total"]
flag_summary["pct_flag_iqr30"] = 100 * flag_summary["n_flag_iqr30"] / flag_summary["n_total"]

flag_summary = flag_summary.sort_values(["pct_flag_iqr15", "n_flag_iqr15"], ascending=False)

# -------------------------------------------------
# 8) Guardar auditoría
# -------------------------------------------------
station_summary_path = OUT_DIR / "HBM_PM25_obs_audit_station_summary.csv"
duplicates_path = OUT_DIR / "HBM_PM25_obs_audit_duplicates.csv"
flagged_rows_path = OUT_DIR / "HBM_PM25_obs_audit_flagged_rows.csv"
flag_summary_path = OUT_DIR / "HBM_PM25_obs_audit_flag_summary.csv"

station_summary.to_csv(station_summary_path, index=False)
dup_rows.to_csv(duplicates_path, index=False)
flagged_rows.to_csv(flagged_rows_path, index=False)
flag_summary.to_csv(flag_summary_path, index=False)

# -------------------------------------------------
# 9) Mostrar resultados
# -------------------------------------------------
print("\nResumen por estación:")
display(station_summary)

print("\nResumen de observaciones atípicas por estación:")
display(flag_summary)

print("\nPrimeras 20 observaciones sospechosas (IQR 1.5):")
display(flagged_rows.head(20))

print("\nArchivos guardados:")
print("-", station_summary_path)
print("-", duplicates_path)
print("-", flagged_rows_path)
print("-", flag_summary_path)

Resumen general del panel observado PM2.5
- filas totales en archivo            : 862
- filas válidas PM2.5                : 808
- estaciones con PM2.5 válido        : 16
- meses observados PM2.5             : 60
- rango temporal                     : 2020-01-01 a 2024-12-01

Duplicados por estación-mes:
- número de filas duplicadas: 0
- número de combinaciones duplicadas: 0

Resumen por estación:


,Estacion,n,mean,std,min,q01,q05,q25,median,q75,q95,q99,max
1,Carvajal - Sevillana,36,32.969860,6.911568,18.519718,18.991463,21.891855,27.975281,33.446371,37.629680,42.572293,46.682765,48.271889
7,Kennedy,54,21.213496,5.887640,9.075212,9.893371,12.333130,17.833018,20.894828,24.548958,30.309963,35.279183,38.651700
10,Movil Fontibon,43,20.945950,5.062750,10.429802,11.213605,13.211790,18.279614,20.148886,24.486874,27.826976,32.805619,36.261831
5,Fontibon,57,19.169282,5.366224,8.539245,9.909468,11.846008,15.504494,18.777860,21.449919,27.255827,35.393145,37.472482
3,Ciudad Bolivar,12,17.636170,4.845790,10.472881,10.501898,10.617966,13.744895,19.173946,21.925982,23.095106,23.107436,23.110518
11,Puente Aranda,59,17.144319,6.034727,5.104018,5.149119,7.940079,12.753140,16.808438,20.026076,27.258921,31.932485,37.357021
14,Tunal,58,16.788691,6.151187,6.188034,6.763960,7.651699,11.490155,17.112077,21.046759,25.505038,30.824473,31.978903
0,Bolivia,42,14.844154,4.078052,4.576040,5.670759,7.356603,12.234956,15.504561,18.162863,20.302612,21.496061,21.616190
13,Suba,58,14.739178,5.537005,6.642793,6.887230,7.671638,10.895217,13.957181,16.816879,23.548628,31.561347,33.220651
8,Las Ferias,57,14.495973,5.497687,6.011130,6.120674,6.873497,10.398105,14.200282,17.305022,23.589098,31.434328,34.273371



Resumen de observaciones atípicas por estación:


,Estacion,n_total,n_flag_iqr15,n_flag_iqr30,pm25_mean,pm25_std,pm25_min,pm25_max,pct_flag_iqr15,pct_flag_iqr30
5,Fontibon,57,2,0,19.169282,5.366224,8.539245,37.472482,3.508772,0.0
8,Las Ferias,57,2,0,14.495973,5.497687,6.011130,34.273371,3.508772,0.0
13,Suba,58,2,0,14.739178,5.537005,6.642793,33.220651,3.448276,0.0
6,Guaymaral,59,2,0,13.827368,4.932624,5.801600,29.789548,3.389831,0.0
9,MinAmbiente,59,2,0,13.737155,5.398499,5.135922,29.323488,3.389831,0.0
4,Colina,42,1,0,10.663401,4.479673,2.702687,24.987714,2.380952,0.0
10,Movil Fontibon,43,1,0,20.945950,5.062750,10.429802,36.261831,2.325581,0.0
7,Kennedy,54,1,0,21.213496,5.887640,9.075212,38.651700,1.851852,0.0
15,Usaquen,54,1,0,11.661781,5.533421,3.565640,33.310408,1.851852,0.0
2,Centro de Alto Rendimiento,59,1,0,13.553019,5.367825,4.673699,32.578869,1.694915,0.0



Primeras 20 observaciones sospechosas (IQR 1.5):


,Estacion,fecha,Año,Mes,PM25_obs,Temp_media,HR,Vel_viento,Presión,Precipitación,q1,q3,iqr,lower_iqr15,upper_iqr15,flag_iqr15,flag_iqr30
25,Centro de Alto Rendimiento,2020-03-01,2020,3,32.578869,19.381613,80.223226,1.050323,NaN,106.01,10.061371,16.862431,6.801059,-0.140218,27.064020,True,False
672,Colina,2024-03-01,2024,3,24.987714,21.388387,71.983226,1.159677,NaN,80.08,7.161553,12.848246,5.686693,-1.368486,21.378285,True,False
26,Fontibon,2020-03-01,2020,3,37.472482,19.381613,80.223226,1.050323,NaN,106.01,15.504494,21.449919,5.945425,6.586357,30.368056,True,False
673,Fontibon,2024-03-01,2024,3,33.759379,21.388387,71.983226,1.159677,NaN,80.08,15.504494,21.449919,5.945425,6.586357,30.368056,True,False
27,Guaymaral,2020-03-01,2020,3,29.789548,14.388065,85.819032,0.820323,NaN,121.62,10.438926,16.502743,6.063817,1.343201,25.598468,True,False
674,Guaymaral,2024-03-01,2024,3,28.917018,15.740000,83.941290,1.051290,NaN,92.23,10.438926,16.502743,6.063817,1.343201,25.598468,True,False
28,Kennedy,2020-03-01,2020,3,38.651700,19.381613,80.223226,1.050323,NaN,106.01,17.833018,24.548958,6.715940,7.759109,34.622867,True,False
29,Las Ferias,2020-03-01,2020,3,34.273371,19.381613,80.223226,1.050323,NaN,106.01,10.398105,17.305022,6.906917,0.037730,27.665396,True,False
676,Las Ferias,2024-03-01,2024,3,29.203652,21.388387,71.983226,1.159677,NaN,80.08,10.398105,17.305022,6.906917,0.037730,27.665396,True,False
30,MinAmbiente,2020-03-01,2020,3,29.034046,19.381613,80.223226,1.050323,NaN,106.01,9.485772,16.987383,7.501611,-1.766644,28.239798,True,False



Archivos guardados:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_obs_audit_station_summary.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_obs_audit_duplicates.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_obs_audit_flagged_rows.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_obs_audit_flag_summary.csv


# CELDA 9A — Auditoría de calidad del panel observado PM2.5

 **Objetivo:**
 revisar la calidad del panel observado de estaciones antes de hacer limpieza
 o volver a ajustar el HBM.

**Qué hace esta celda:**
 1. Lee el archivo corregido de estaciones.
 2. Construye la fecha mensual.
 3. Filtra observaciones válidas de PM2.5.
 4. Revisa duplicados por estación y mes.
 5. Resume la distribución de PM2.5 por estación:
    - número de observaciones
    - media
    - desviación estándar
    - percentiles
    - mínimo y máximo
 6. Marca observaciones atípicas por estación usando una regla robusta IQR.
 7. Genera una tabla con las filas sospechosas.
 8. Guarda archivos de auditoría para usarlos en la siguiente etapa de limpieza.

 **Importante:**
 esta celda no elimina datos.
 Solo identifica dónde puede estar entrando ruido al modelo.

In [11]:
# %%
# =========================
# CELDA 9B) Auditoría de covariables de malla PM2.5
# =========================

grid_audit = pd.read_csv(GRID_CSV)
grid_audit["fecha"] = pd.to_datetime(grid_audit["fecha"], errors="coerce")

if grid_audit["fecha"].isna().any():
    raise ValueError("Hay fechas inválidas en grid_3km_AD_mensual_2020_2024.csv.")

covars_pm25 = [
    "Vel_viento_idw",
    "diff_PM25_grid",
    "grad_PM25_grid",
    "adv_proxy_PM25_grid",
]

missing_cols = [c for c in covars_pm25 if c not in grid_audit.columns]
if missing_cols:
    raise ValueError(f"Faltan columnas en la malla: {missing_cols}")

grid_audit["year"] = grid_audit["fecha"].dt.year
grid_audit["month"] = grid_audit["fecha"].dt.month

print("Resumen general de la malla auditada")
print("- filas totales       :", len(grid_audit))
print("- celdas únicas       :", grid_audit["cell_id"].nunique())
print("- meses únicos        :", grid_audit["fecha"].nunique())
print("- rango temporal      :", grid_audit["fecha"].min().date(), "a", grid_audit["fecha"].max().date())

# -------------------------------------------------
# 1) Resumen descriptivo de covariables
# -------------------------------------------------
desc_rows = []

for col in covars_pm25:
    s = pd.to_numeric(grid_audit[col], errors="coerce")
    desc_rows.append({
        "variable": col,
        "n": int(s.notna().sum()),
        "n_null": int(s.isna().sum()),
        "n_inf": int(np.isinf(s).sum()),
        "mean": float(np.nanmean(s)),
        "std": float(np.nanstd(s, ddof=1)),
        "min": float(np.nanmin(s)),
        "q01": float(np.nanquantile(s, 0.01)),
        "q05": float(np.nanquantile(s, 0.05)),
        "q25": float(np.nanquantile(s, 0.25)),
        "median": float(np.nanquantile(s, 0.50)),
        "q75": float(np.nanquantile(s, 0.75)),
        "q95": float(np.nanquantile(s, 0.95)),
        "q99": float(np.nanquantile(s, 0.99)),
        "max": float(np.nanmax(s)),
    })

desc_covars = pd.DataFrame(desc_rows)

# -------------------------------------------------
# 2) Flags IQR por covariable
# -------------------------------------------------
flag_rows = []
extreme_tables = []
monthly_flag_tables = []

for col in covars_pm25:
    tmp = grid_audit[["cell_id", "fecha", "year", "month", col]].copy()
    tmp[col] = pd.to_numeric(tmp[col], errors="coerce")

    q1 = tmp[col].quantile(0.25)
    q3 = tmp[col].quantile(0.75)
    iqr = q3 - q1

    lower_15 = q1 - 1.5 * iqr
    upper_15 = q3 + 1.5 * iqr
    lower_30 = q1 - 3.0 * iqr
    upper_30 = q3 + 3.0 * iqr

    tmp["variable"] = col
    tmp["q1"] = q1
    tmp["q3"] = q3
    tmp["iqr"] = iqr
    tmp["lower_iqr15"] = lower_15
    tmp["upper_iqr15"] = upper_15
    tmp["lower_iqr30"] = lower_30
    tmp["upper_iqr30"] = upper_30

    tmp["flag_iqr15"] = (tmp[col] < lower_15) | (tmp[col] > upper_15)
    tmp["flag_iqr30"] = (tmp[col] < lower_30) | (tmp[col] > upper_30)

    n_total = tmp[col].notna().sum()
    n_flag15 = int(tmp["flag_iqr15"].sum())
    n_flag30 = int(tmp["flag_iqr30"].sum())

    flag_rows.append({
        "variable": col,
        "n_total": int(n_total),
        "n_flag_iqr15": n_flag15,
        "n_flag_iqr30": n_flag30,
        "pct_flag_iqr15": 100 * n_flag15 / n_total if n_total else np.nan,
        "pct_flag_iqr30": 100 * n_flag30 / n_total if n_total else np.nan,
        "q1": float(q1),
        "q3": float(q3),
        "iqr": float(iqr),
        "lower_iqr15": float(lower_15),
        "upper_iqr15": float(upper_15),
        "lower_iqr30": float(lower_30),
        "upper_iqr30": float(upper_30),
    })

    # extremos por valor absoluto
    tmp["abs_value"] = tmp[col].abs()
    top_extreme = (
        tmp.sort_values("abs_value", ascending=False)
        .head(15)
        .copy()
    )
    extreme_tables.append(top_extreme)

    # resumen mensual de flags
    monthly_flags = (
        tmp.groupby(["year", "month"])
        .agg(
            n_total=(col, "count"),
            n_flag_iqr15=("flag_iqr15", "sum"),
            n_flag_iqr30=("flag_iqr30", "sum"),
            mean_value=(col, "mean"),
            std_value=(col, "std"),
        )
        .reset_index()
    )
    monthly_flags["variable"] = col
    monthly_flags["pct_flag_iqr15"] = 100 * monthly_flags["n_flag_iqr15"] / monthly_flags["n_total"]
    monthly_flags["pct_flag_iqr30"] = 100 * monthly_flags["n_flag_iqr30"] / monthly_flags["n_total"]
    monthly_flag_tables.append(monthly_flags)

flag_summary_covars = pd.DataFrame(flag_rows).sort_values("pct_flag_iqr15", ascending=False)
extreme_covars = pd.concat(extreme_tables, ignore_index=True)
monthly_flags_covars = pd.concat(monthly_flag_tables, ignore_index=True)

# -------------------------------------------------
# 3) Correlación entre covariables
# -------------------------------------------------
corr_covars = grid_audit[covars_pm25].corr(numeric_only=True)

# -------------------------------------------------
# 4) Guardar auditoría
# -------------------------------------------------
desc_covars_path = OUT_DIR / "HBM_PM25_grid_audit_covariate_summary.csv"
flag_covars_path = OUT_DIR / "HBM_PM25_grid_audit_flag_summary.csv"
extreme_covars_path = OUT_DIR / "HBM_PM25_grid_audit_extreme_rows.csv"
monthly_flags_covars_path = OUT_DIR / "HBM_PM25_grid_audit_monthly_flags.csv"
corr_covars_path = OUT_DIR / "HBM_PM25_grid_audit_correlation.csv"

desc_covars.to_csv(desc_covars_path, index=False)
flag_summary_covars.to_csv(flag_covars_path, index=False)
extreme_covars.to_csv(extreme_covars_path, index=False)
monthly_flags_covars.to_csv(monthly_flags_covars_path, index=False)
corr_covars.to_csv(corr_covars_path)

# -------------------------------------------------
# 5) Mostrar resultados
# -------------------------------------------------
print("\nResumen descriptivo de covariables:")
display(desc_covars)

print("\nResumen de flags por covariable:")
display(flag_summary_covars)

print("\nCorrelación entre covariables:")
display(corr_covars)

print("\nCasos más extremos por valor absoluto:")
display(
    extreme_covars[
        ["variable", "cell_id", "fecha", "year", "month"] + covars_pm25
    ].head(30)
)

print("\nMeses con mayor porcentaje de flags IQR 1.5 por covariable:")
top_months_flags = (
    monthly_flags_covars
    .sort_values(["variable", "pct_flag_iqr15"], ascending=[True, False])
    .groupby("variable")
    .head(10)
    .reset_index(drop=True)
)
display(top_months_flags)

print("\nArchivos guardados:")
print("-", desc_covars_path)
print("-", flag_covars_path)
print("-", extreme_covars_path)
print("-", monthly_flags_covars_path)
print("-", corr_covars_path)

Resumen general de la malla auditada
- filas totales       : 15240
- celdas únicas       : 254
- meses únicos        : 60
- rango temporal      : 2020-01-01 a 2024-12-01

Resumen descriptivo de covariables:


,variable,n,n_null,n_inf,mean,std,min,q01,q05,q25,median,q75,q95,q99,max
0,Vel_viento_idw,15240,0,0,1.116795e+00,0.221991,5.046784e-01,8.076667e-01,0.869237,0.944333,1.061290,1.215806,1.591290,1.717742,1.717742
1,diff_PM25_grid,15240,0,0,-7.809443e-18,3.790050,-5.177307e+01,-1.420085e+01,-2.696648,-0.296371,0.000297,0.086797,3.807190,13.310876,34.832500
2,grad_PM25_grid,15240,0,0,1.216713e-04,0.000220,4.927967e-09,4.796772e-07,0.000001,0.000007,0.000036,0.000140,0.000513,0.001092,0.003206
3,adv_proxy_PM25_grid,15240,0,0,1.356052e-04,0.000252,4.492399e-09,4.898759e-07,0.000001,0.000008,0.000039,0.000153,0.000582,0.001246,0.004107



Resumen de flags por covariable:


,variable,n_total,n_flag_iqr15,n_flag_iqr30,pct_flag_iqr15,pct_flag_iqr30,q1,q3,iqr,lower_iqr15,upper_iqr15,lower_iqr30,upper_iqr30
1,diff_PM25_grid,15240,4425,3107,29.035433,20.387139,-0.296371,0.086797,0.383168,-0.871122,0.661548,-1.445874,1.236300
3,adv_proxy_PM25_grid,15240,1622,746,10.643045,4.895013,0.000008,0.000153,0.000145,-0.000210,0.000371,-0.000428,0.000589
2,grad_PM25_grid,15240,1563,669,10.255906,4.389764,0.000007,0.000140,0.000133,-0.000193,0.000341,-0.000393,0.000541
0,Vel_viento_idw,15240,500,0,3.280840,0.000000,0.944333,1.215806,0.271473,0.537124,1.623016,0.129914,2.030226



Correlación entre covariables:


...


Casos más extremos por valor absoluto:


,variable,cell_id,fecha,year,month,Vel_viento_idw,diff_PM25_grid,grad_PM25_grid,adv_proxy_PM25_grid
0,Vel_viento_idw,486,2023-07-01,2023,7,1.717742,NaN,NaN,NaN
1,Vel_viento_idw,464,2023-07-01,2023,7,1.717742,NaN,NaN,NaN
2,Vel_viento_idw,523,2023-07-01,2023,7,1.717742,NaN,NaN,NaN
3,Vel_viento_idw,304,2023-07-01,2023,7,1.717742,NaN,NaN,NaN
4,Vel_viento_idw,126,2023-07-01,2023,7,1.717742,NaN,NaN,NaN
5,Vel_viento_idw,169,2023-07-01,2023,7,1.717742,NaN,NaN,NaN
6,Vel_viento_idw,176,2023-07-01,2023,7,1.717742,NaN,NaN,NaN
7,Vel_viento_idw,2,2023-07-01,2023,7,1.717742,NaN,NaN,NaN
8,Vel_viento_idw,517,2023-07-01,2023,7,1.717742,NaN,NaN,NaN
9,Vel_viento_idw,519,2023-07-01,2023,7,1.717742,NaN,NaN,NaN



Meses con mayor porcentaje de flags IQR 1.5 por covariable:


,year,month,n_total,n_flag_iqr15,n_flag_iqr30,mean_value,std_value,variable,pct_flag_iqr15,pct_flag_iqr30
0,2023,7,254,254,0,1.713691e+00,0.014138,Vel_viento_idw,100.000000,0.000000
1,2021,7,254,241,0,1.637227e+00,0.006300,Vel_viento_idw,94.881890,0.000000
2,2020,11,254,5,0,7.889624e-01,0.059452,Vel_viento_idw,1.968504,0.000000
3,2020,1,254,0,0,1.480073e+00,0.073592,Vel_viento_idw,0.000000,0.000000
4,2020,2,254,0,0,1.350505e+00,0.035660,Vel_viento_idw,0.000000,0.000000
5,2020,3,254,0,0,1.037031e+00,0.042247,Vel_viento_idw,0.000000,0.000000
6,2020,4,254,0,0,8.910268e-01,0.002204,Vel_viento_idw,0.000000,0.000000
7,2020,5,254,0,0,1.143411e+00,0.002489,Vel_viento_idw,0.000000,0.000000
8,2020,6,254,0,0,1.243268e+00,0.002327,Vel_viento_idw,0.000000,0.000000
9,2020,7,254,0,0,1.134478e+00,0.010132,Vel_viento_idw,0.000000,0.000000



Archivos guardados:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_grid_audit_covariate_summary.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_grid_audit_flag_summary.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_grid_audit_extreme_rows.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_grid_audit_monthly_flags.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_grid_audit_correlation.csv



 # CELDA 9C — Construcción de covariables limpias para el HBM de PM2.5

 **Objetivo:**
 crear una versión depurada y más estable de las covariables de malla
 para volver a correr el HBM con menor sensibilidad a colas extremas
 y menor redundancia entre predictores.

 **Estrategia aplicada:**
 - `Vel_viento_idw`: se conserva casi intacta
 - `diff_PM25_grid`: se transforma con signed-log y luego se winsoriza
 - `grad_PM25_grid`: se winsoriza solo para auditoría, pero no será usada
- `adv_proxy_PM25_grid`: se winsoriza y se conserva para el modelo

 **Variables finales recomendadas para el nuevo HBM:**
 - `Vel_viento_idw_clean`
 - `diff_PM25_grid_slog_clean`
 - `adv_proxy_PM25_grid_clean`

 **Importante:**
 en esta etapa no se reentrena el HBM.
 Solo se construye y guarda la malla limpia.

In [12]:
# %%
# =========================
# CELDA 9C) Construcción de covariables limpias PM2.5
# =========================

grid_clean = grid_audit.copy()

def winsorize_series(s, q_low=0.01, q_high=0.99):
    s = pd.to_numeric(s, errors="coerce").astype(float)
    lo = s.quantile(q_low)
    hi = s.quantile(q_high)
    s_clip = s.clip(lower=lo, upper=hi)
    return s_clip, lo, hi

transform_report = []

# -------------------------------------------------
# 1) Velocidad del viento: conservar casi intacta
# -------------------------------------------------
grid_clean["Vel_viento_idw_clean"] = pd.to_numeric(grid_clean["Vel_viento_idw"], errors="coerce").astype(float)

transform_report.append({
    "variable_original": "Vel_viento_idw",
    "variable_limpia": "Vel_viento_idw_clean",
    "transformacion": "sin cambio",
    "q_low": np.nan,
    "q_high": np.nan
})

# -------------------------------------------------
# 2) diff_PM25_grid: signed-log + winsorización
# -------------------------------------------------
x_diff = pd.to_numeric(grid_clean["diff_PM25_grid"], errors="coerce").astype(float)
grid_clean["diff_PM25_grid_slog"] = np.sign(x_diff) * np.log1p(np.abs(x_diff))

grid_clean["diff_PM25_grid_slog_clean"], lo_diff, hi_diff = winsorize_series(
    grid_clean["diff_PM25_grid_slog"], q_low=0.01, q_high=0.99
)

transform_report.append({
    "variable_original": "diff_PM25_grid",
    "variable_limpia": "diff_PM25_grid_slog_clean",
    "transformacion": "signed_log1p + winsor_1_99",
    "q_low": float(lo_diff),
    "q_high": float(hi_diff)
})

# -------------------------------------------------
# 3) grad_PM25_grid: winsorización (solo auditoría / respaldo)
# -------------------------------------------------
grid_clean["grad_PM25_grid_clean"], lo_grad, hi_grad = winsorize_series(
    grid_clean["grad_PM25_grid"], q_low=0.01, q_high=0.99
)

transform_report.append({
    "variable_original": "grad_PM25_grid",
    "variable_limpia": "grad_PM25_grid_clean",
    "transformacion": "winsor_1_99",
    "q_low": float(lo_grad),
    "q_high": float(hi_grad)
})

# -------------------------------------------------
# 4) adv_proxy_PM25_grid: winsorización
# -------------------------------------------------
grid_clean["adv_proxy_PM25_grid_clean"], lo_adv, hi_adv = winsorize_series(
    grid_clean["adv_proxy_PM25_grid"], q_low=0.01, q_high=0.99
)

transform_report.append({
    "variable_original": "adv_proxy_PM25_grid",
    "variable_limpia": "adv_proxy_PM25_grid_clean",
    "transformacion": "winsor_1_99",
    "q_low": float(lo_adv),
    "q_high": float(hi_adv)
})

transform_report_df = pd.DataFrame(transform_report)

# -------------------------------------------------
# 5) Definir covariables recomendadas para el nuevo HBM
# -------------------------------------------------
X_COLS_PM25_CLEAN = [
    "Vel_viento_idw_clean",
    "diff_PM25_grid_slog_clean",
    "adv_proxy_PM25_grid_clean",
]

# -------------------------------------------------
# 6) Guardar archivo limpio
# -------------------------------------------------
grid_clean_path = OUT_DIR / "HBM_PM25_grid_clean_v1.csv"
transform_report_path = OUT_DIR / "HBM_PM25_grid_clean_transform_report.csv"
xcols_clean_path = OUT_DIR / "HBM_PM25_grid_clean_xcols.json"

grid_clean.to_csv(grid_clean_path, index=False)
transform_report_df.to_csv(transform_report_path, index=False)

with open(xcols_clean_path, "w", encoding="utf-8") as f:
    json.dump(X_COLS_PM25_CLEAN, f, ensure_ascii=False, indent=2)

# -------------------------------------------------
# 7) Resumen comparativo
# -------------------------------------------------
compare_summary = pd.DataFrame({
    "variable": [
        "Vel_viento_idw",
        "diff_PM25_grid",
        "diff_PM25_grid_slog_clean",
        "grad_PM25_grid",
        "grad_PM25_grid_clean",
        "adv_proxy_PM25_grid",
        "adv_proxy_PM25_grid_clean",
    ],
    "mean": [
        grid_clean["Vel_viento_idw"].mean(),
        grid_clean["diff_PM25_grid"].mean(),
        grid_clean["diff_PM25_grid_slog_clean"].mean(),
        grid_clean["grad_PM25_grid"].mean(),
        grid_clean["grad_PM25_grid_clean"].mean(),
        grid_clean["adv_proxy_PM25_grid"].mean(),
        grid_clean["adv_proxy_PM25_grid_clean"].mean(),
    ],
    "std": [
        grid_clean["Vel_viento_idw"].std(),
        grid_clean["diff_PM25_grid"].std(),
        grid_clean["diff_PM25_grid_slog_clean"].std(),
        grid_clean["grad_PM25_grid"].std(),
        grid_clean["grad_PM25_grid_clean"].std(),
        grid_clean["adv_proxy_PM25_grid"].std(),
        grid_clean["adv_proxy_PM25_grid_clean"].std(),
    ],
    "min": [
        grid_clean["Vel_viento_idw"].min(),
        grid_clean["diff_PM25_grid"].min(),
        grid_clean["diff_PM25_grid_slog_clean"].min(),
        grid_clean["grad_PM25_grid"].min(),
        grid_clean["grad_PM25_grid_clean"].min(),
        grid_clean["adv_proxy_PM25_grid"].min(),
        grid_clean["adv_proxy_PM25_grid_clean"].min(),
    ],
    "max": [
        grid_clean["Vel_viento_idw"].max(),
        grid_clean["diff_PM25_grid"].max(),
        grid_clean["diff_PM25_grid_slog_clean"].max(),
        grid_clean["grad_PM25_grid"].max(),
        grid_clean["grad_PM25_grid_clean"].max(),
        grid_clean["adv_proxy_PM25_grid"].max(),
        grid_clean["adv_proxy_PM25_grid_clean"].max(),
    ],
})

print("Covariables limpias construidas correctamente.\n")

print("Variables recomendadas para el nuevo HBM:")
print(X_COLS_PM25_CLEAN)

print("\nReporte de transformaciones:")
display(transform_report_df)

print("\nResumen comparativo antes/después:")
display(compare_summary)

print("\nArchivos guardados:")
print("-", grid_clean_path)
print("-", transform_report_path)
print("-", xcols_clean_path)

Covariables limpias construidas correctamente.

Variables recomendadas para el nuevo HBM:
['Vel_viento_idw_clean', 'diff_PM25_grid_slog_clean', 'adv_proxy_PM25_grid_clean']

Reporte de transformaciones:


,variable_original,variable_limpia,transformacion,q_low,q_high
0,Vel_viento_idw,Vel_viento_idw_clean,sin cambio,NaN,NaN
1,diff_PM25_grid,diff_PM25_grid_slog_clean,signed_log1p + winsor_1_99,-2.721351e+00,2.661017
2,grad_PM25_grid,grad_PM25_grid_clean,winsor_1_99,4.796772e-07,0.001092
3,adv_proxy_PM25_grid,adv_proxy_PM25_grid_clean,winsor_1_99,4.898759e-07,0.001246



Resumen comparativo antes/después:


,variable,mean,std,min,max
0,Vel_viento_idw,1.116795e+00,0.221991,5.046784e-01,1.717742
1,diff_PM25_grid,-7.809443e-18,3.790050,-5.177307e+01,34.832500
2,diff_PM25_grid_slog_clean,-2.427159e-02,0.817757,-2.721351e+00,2.661017
3,grad_PM25_grid,1.216713e-04,0.000220,4.927967e-09,0.003206
4,grad_PM25_grid_clean,1.172720e-04,0.000191,4.796772e-07,0.001092
5,adv_proxy_PM25_grid,1.356052e-04,0.000252,4.492399e-09,0.004107
6,adv_proxy_PM25_grid_clean,1.304616e-04,0.000216,4.898759e-07,0.001246



Archivos guardados:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_grid_clean_v1.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_grid_clean_transform_report.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_grid_clean_xcols.json



# CELDA 9D — Reconstrucción del panel HBM con covariables limpias

 **Objetivo:**
 construir la nueva base de modelación del HBM para PM2.5 usando
 las covariables limpias generadas en la celda anterior.

 **Qué hace esta celda:**
 1. Toma la malla limpia `grid_clean`.
 2. Construye `grid_base_clean` con:
    - `cell_id`
    - `fecha`
    - `year`
    - `month`
    - `cell_idx`
    - `time_idx`
    - covariables limpias recomendadas
 3. Reconstruye el panel observado válido de PM2.5.
 4. Une observaciones reales con la malla limpia por:
    - `cell_id`
    - `fecha`
 5. Guarda los archivos base para volver a correr el HBM limpio.

 **Salidas principales:**
 - `HBM_PM25_grid_base_clean_v1.csv`
 - `HBM_PM25_obs_panel_clean_v1.csv`
 - `HBM_PM25_clean_xcols.json`

In [13]:
# %%
# =========================
# CELDA 9D) Reconstruir panel HBM con covariables limpias
# =========================

# -------------------------------------------------
# 1) Verificaciones mínimas
# -------------------------------------------------
required_clean_cols = [
    "cell_id", "fecha",
    "Vel_viento_idw_clean",
    "diff_PM25_grid_slog_clean",
    "adv_proxy_PM25_grid_clean",
]

missing_clean = [c for c in required_clean_cols if c not in grid_clean.columns]
if missing_clean:
    raise ValueError(f"Faltan columnas en grid_clean: {missing_clean}")

# asegurar fecha
grid_clean["fecha"] = pd.to_datetime(grid_clean["fecha"], errors="coerce")
if grid_clean["fecha"].isna().any():
    raise ValueError("Hay fechas inválidas en grid_clean.")

grid_clean["year"] = grid_clean["fecha"].dt.year
grid_clean["month"] = grid_clean["fecha"].dt.month

# -------------------------------------------------
# 2) Construir grid_base_clean
# -------------------------------------------------
grid_base_clean = grid_clean[
    ["cell_id", "fecha", "year", "month"] + X_COLS_PM25_CLEAN
].copy()

# usar los mismos índices globales de celda y tiempo
grid_base_clean["cell_idx"] = grid_base_clean["cell_id"].map(cell_map)
grid_base_clean["time_idx"] = grid_base_clean["fecha"].map(time_map)

if grid_base_clean["cell_idx"].isna().any():
    raise ValueError("Hay cell_id en la malla limpia que no pudieron mapearse con cell_map.")
if grid_base_clean["time_idx"].isna().any():
    raise ValueError("Hay fechas en la malla limpia que no pudieron mapearse con time_map.")

grid_base_clean["cell_idx"] = grid_base_clean["cell_idx"].astype(int)
grid_base_clean["time_idx"] = grid_base_clean["time_idx"].astype(int)

# -------------------------------------------------
# 3) Reconstruir observaciones válidas PM2.5
# -------------------------------------------------
obs_clean_src = pd.read_csv(OBS_CSV)

obs_clean_src["fecha"] = pd.to_datetime(
    dict(year=obs_clean_src["Año"].astype(int), month=obs_clean_src["Mes"].astype(int), day=1),
    errors="coerce"
)

if obs_clean_src["fecha"].isna().any():
    raise ValueError("Hay fechas inválidas en el panel observado.")

obs_clean_src["valido_PM25"] = obs_clean_src["valido_PM25"].astype(bool)

obs_pm_clean = obs_clean_src.loc[
    (obs_clean_src["valido_PM25"] == True) &
    (obs_clean_src["PM25"].notna()) &
    (obs_clean_src["cell_id"].notna())
].copy()

obs_pm_clean = obs_pm_clean.rename(columns={"PM25": "PM25_obs"})

# -------------------------------------------------
# 4) Unir observaciones con covariables limpias
# -------------------------------------------------
obs_panel_clean = obs_pm_clean.merge(
    grid_base_clean,
    on=["cell_id", "fecha"],
    how="left",
    validate="many_to_one"
)

missing_covs_clean = obs_panel_clean[X_COLS_PM25_CLEAN].isna().any(axis=1).sum()
if missing_covs_clean > 0:
    raise ValueError(
        f"Hay {missing_covs_clean} observaciones sin covariables limpias. "
        "Revisa cell_id y fecha."
    )

# -------------------------------------------------
# 5) Guardar archivos base limpios
# -------------------------------------------------
grid_base_clean_path = OUT_DIR / "HBM_PM25_grid_base_clean_v1.csv"
obs_panel_clean_path = OUT_DIR / "HBM_PM25_obs_panel_clean_v1.csv"
xcols_clean_path = OUT_DIR / "HBM_PM25_clean_xcols.json"

grid_base_clean.to_csv(grid_base_clean_path, index=False)
obs_panel_clean.to_csv(obs_panel_clean_path, index=False)

with open(xcols_clean_path, "w", encoding="utf-8") as f:
    json.dump(X_COLS_PM25_CLEAN, f, ensure_ascii=False, indent=2)

# -------------------------------------------------
# 6) Resumen
# -------------------------------------------------
print("Panel HBM limpio reconstruido correctamente.\n")

print("grid_base_clean:")
print("- filas            :", len(grid_base_clean))
print("- celdas únicas    :", grid_base_clean["cell_id"].nunique())
print("- meses únicos     :", grid_base_clean["fecha"].nunique())

print("\nobs_panel_clean:")
print("- filas            :", len(obs_panel_clean))
print("- estaciones únicas:", obs_panel_clean["Estacion"].nunique())
print("- celdas observadas:", obs_panel_clean["cell_id"].nunique())

print("\nCovariables limpias usadas:")
print(X_COLS_PM25_CLEAN)

print("\nArchivos guardados:")
print("-", grid_base_clean_path)
print("-", obs_panel_clean_path)
print("-", xcols_clean_path)

print("\nPrimeras filas de obs_panel_clean:")
display(
    obs_panel_clean[
        ["Estacion", "fecha", "PM25_obs", "cell_id"] + X_COLS_PM25_CLEAN
    ].head()
)

Panel HBM limpio reconstruido correctamente.

grid_base_clean:
- filas            : 15240
- celdas únicas    : 254
- meses únicos     : 60

obs_panel_clean:
- filas            : 808
- estaciones únicas: 16
- celdas observadas: 14

Covariables limpias usadas:
['Vel_viento_idw_clean', 'diff_PM25_grid_slog_clean', 'adv_proxy_PM25_grid_clean']

Archivos guardados:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_grid_base_clean_v1.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_obs_panel_clean_v1.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_clean_xcols.json

Primeras filas de obs_panel_clean:


,Estacion,fecha,PM25_obs,cell_id,Vel_viento_idw_clean,diff_PM25_grid_slog_clean,adv_proxy_PM25_grid_clean
0,Carvajal - Sevillana,2020-01-01,29.811594,482,1.503226,2.466040,0.000841
1,Centro de Alto Rendimiento,2020-01-01,13.672938,567,1.503226,2.661017,0.000823
2,Fontibon,2020-01-01,19.646154,485,1.503226,-1.997473,0.000469
3,Guaymaral,2020-01-01,14.772134,653,1.128177,1.281969,0.000108
4,Kennedy,2020-01-01,23.648442,442,1.503226,-0.981349,0.000868



# CELDA 9E — Validación temporal del HBM limpio (M1b-clean)

 **Objetivo:**
 volver a correr la validación temporal del HBM usando la versión limpia
 de las covariables de malla para PM2.5.

 **Modelo usado:**
 - estructura: `M1b`
 - espacial: `ICAR`
 - temporal: `RW1`
 - observación: `PM25_obs`

 **Covariables limpias:**
 - `Vel_viento_idw_clean`
 - `diff_PM25_grid_slog_clean`
 - `adv_proxy_PM25_grid_clean`

 **Qué hace esta celda:**
 1. Usa `grid_base_clean` y `obs_panel_clean`.
 2. Repite los 3 folds temporales.
 3. Escala covariables usando solo train en cada fold.
 4. Ajusta `fit_hbm_m1b_fold`.
 5. Guarda predicciones, summary y parámetros de escalamiento.
 6. Consolida métricas.
 7. Compara contra el modelo previo `M1b`.

 **Salidas principales:**
 - `HBM_PM25_M1bclean_fold_1_pred_test.csv`
 - `HBM_PM25_M1bclean_fold_2_pred_test.csv`
 - `HBM_PM25_M1bclean_fold_3_pred_test.csv`
 - `HBM_PM25_M1bclean_metrics_folds.csv`
 - `HBM_PM25_compare_M1b_vs_M1bclean.csv`

 **Importante:**
 aquí lo que más nos interesa revisar después es si bajan:
 - `width_90_mean`
 - `wis_90`
 sin deteriorar demasiado MAE/RMSE.

In [17]:
# %%
# =========================
# CELDA 9E) Validación temporal HBM limpio
# =========================

all_metrics_m1bclean = []
all_test_preds_m1bclean = []

DRAWS_C = 1000
TUNE_C = 1500
CHAINS_C = 4
TARGET_ACCEPT_C = 0.99
MAX_TREEDEPTH_C = 15
RANDOM_SEED_C = 42

for fold_name, fold_info in FOLDS.items():
    print("\n" + "=" * 80)
    print(f"Corriendo M1b-clean - {fold_name}")
    print("Train years:", fold_info["train_years"])
    print("Test years :", fold_info["test_years"])

    # -------------------------------------------------
    # 1) Escalar covariables limpias usando solo train
    # -------------------------------------------------
    grid_scaled_c, scale_params_c = scale_grid_by_train_years(
        grid_df=grid_base_clean,
        x_cols=X_COLS_PM25_CLEAN,
        train_years=fold_info["train_years"]
    )

    # -------------------------------------------------
    # 2) Llevar covariables escaladas al panel observado limpio
    # -------------------------------------------------
    obs_scaled_c = merge_scaled_covariates_to_obs(
        obs_df=obs_panel_clean,
        grid_scaled=grid_scaled_c,
        x_cols=X_COLS_PM25_CLEAN
    )

    # -------------------------------------------------
    # 3) Ajustar modelo del fold
    # -------------------------------------------------
    idata_fold_c, test_pred_fold_c, met_fold_c = fit_hbm_m1b_fold(
        obs_scaled=obs_scaled_c,
        grid_scaled=grid_scaled_c,
        w_edges_df=w_edges,
        x_cols_raw=X_COLS_PM25_CLEAN,
        train_years=fold_info["train_years"],
        test_years=fold_info["test_years"],
        n_cells=len(cell_ids),
        n_times=len(time_ids),
        draws=DRAWS_C,
        tune=TUNE_C,
        chains=CHAINS_C,
        target_accept=TARGET_ACCEPT_C,
        max_treedepth=MAX_TREEDEPTH_C,
        random_seed=RANDOM_SEED_C,
    )

    # -------------------------------------------------
    # 4) Guardar predicciones del fold
    # -------------------------------------------------
    test_pred_fold_c["fold"] = fold_name
    pred_fold_path_c = OUT_DIR / f"HBM_PM25_M1bclean_{fold_name}_pred_test.csv"
    test_pred_fold_c.to_csv(pred_fold_path_c, index=False)

    # -------------------------------------------------
    # 5) Guardar summary del posterior
    # -------------------------------------------------
    summary_fold_c = az.summary(
        idata_fold_c,
        var_names=["alpha", "beta", "sigma_y", "tau_phi", "sigma_t"],
        round_to=4
    )
    summary_fold_path_c = OUT_DIR / f"HBM_PM25_M1bclean_{fold_name}_summary.csv"
    summary_fold_c.to_csv(summary_fold_path_c)

    # -------------------------------------------------
    # 6) Guardar parámetros de escalamiento
    # -------------------------------------------------
    scale_fold_path_c = OUT_DIR / f"HBM_PM25_M1bclean_{fold_name}_scale_params.json"
    with open(scale_fold_path_c, "w", encoding="utf-8") as f:
        json.dump(scale_params_c, f, ensure_ascii=False, indent=2)

    # -------------------------------------------------
    # 7) Consolidar métricas
    # -------------------------------------------------
    met_fold_c["model"] = "M1b_clean"
    met_fold_c["fold"] = fold_name
    met_fold_c["train_years"] = ",".join(map(str, fold_info["train_years"]))
    met_fold_c["test_years"] = ",".join(map(str, fold_info["test_years"]))

    all_metrics_m1bclean.append(met_fold_c)
    all_test_preds_m1bclean.append(test_pred_fold_c)

    print("\nGuardado del fold:")
    print("-", pred_fold_path_c)
    print("-", summary_fold_path_c)
    print("-", scale_fold_path_c)

    print("\nMétricas del fold:")
    for k, v in met_fold_c.items():
        print(f"- {k}: {v}")

# -------------------------------------------------
# 8) Consolidar resultados M1b-clean
# -------------------------------------------------
metrics_m1bclean_df = pd.DataFrame(all_metrics_m1bclean)
preds_m1bclean_df = pd.concat(all_test_preds_m1bclean, ignore_index=True)

metrics_m1bclean_path = OUT_DIR / "HBM_PM25_M1bclean_metrics_folds.csv"
preds_m1bclean_path = OUT_DIR / "HBM_PM25_M1bclean_pred_test_all_folds.csv"

metrics_m1bclean_df.to_csv(metrics_m1bclean_path, index=False)
preds_m1bclean_df.to_csv(preds_m1bclean_path, index=False)

print("\n" + "=" * 80)
print("VALIDACIÓN TEMPORAL M1b-clean COMPLETADA")
print("- métricas consolidadas :", metrics_m1bclean_path)
print("- predicciones consolidadas :", preds_m1bclean_path)

print("\nResumen final de métricas M1b-clean:")
display(metrics_m1bclean_df)

# -------------------------------------------------
# 9) Comparar M1b vs M1b-clean
# -------------------------------------------------
m1b_metrics_path = OUT_DIR / "HBM_PM25_M1b_metrics_folds.csv"

if m1b_metrics_path.exists():
    metrics_m1b_df = pd.read_csv(m1b_metrics_path).copy()
    metrics_m1b_df["model"] = "M1b"

    cols_keep = [
        "model", "fold", "n", "mae", "rmse", "bias", "r", "r2",
        "coverage_90", "width_90_mean", "wis_90", "train_years", "test_years"
    ]

    compare_clean_df = pd.concat(
        [
            metrics_m1b_df[cols_keep],
            metrics_m1bclean_df[cols_keep]
        ],
        ignore_index=True
    )

    compare_clean_path = OUT_DIR / "HBM_PM25_compare_M1b_vs_M1bclean.csv"
    compare_clean_df.to_csv(compare_clean_path, index=False)

    print("\nComparación M1b vs M1b-clean guardada en:")
    print("-", compare_clean_path)

    print("\nTabla comparativa:")
    display(compare_clean_df.sort_values(["fold", "model"]).reset_index(drop=True))
else:
    print("\nNo se encontró el archivo de métricas de M1b previo.")
    print("Se omitió la comparación M1b vs M1b-clean.")


Corriendo M1b-clean - fold_1
Train years: [2020, 2021]
Test years : [2022]


Initializing NUTS using adapt_diag...
ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: local_subtensor_merge
ERROR (pytensor.graph.rewriting.basic): node: Subtensor{i}(Subtensor{start:}.0, 0)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 1920, in process_node
    replacements = node_rewriter.transform(
                   ^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 993, in transform
    return self.fn(fgraph, node)
           ^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 416, in local_subtensor_merge
    merge_two_slices(
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 865, in

Output()

Sampling 4 chains for 1_500 tune and 1_000 draw iterations (6_000 + 4_000 draws total) took 714 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



Guardado del fold:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1bclean_fold_1_pred_test.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1bclean_fold_1_summary.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1bclean_fold_1_scale_params.json

Métricas del fold:
- n: 165
- mae: 4.596562896886717
- rmse: 5.967040210701766
- bias: 3.374788310318431
- r: 0.6633033660934421
- r2: 0.4399713554708909
- coverage_90: 1.0
- width_90_mean: 46.074545443724325
- wis_90: 46.074545443724325
- model: M1b_clean
- fold: fold_1
- train_years: 2020,2021
- test_years: 2022

Corriendo M1b-clean - fold_2
Train years: [2020, 2021, 2022]
Test years : [2023]


...

Sampling 4 chains for 1_500 tune and 1_000 draw iterations (6_000 + 4_000 draws total) took 525 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



Guardado del fold:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1bclean_fold_3_pred_test.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1bclean_fold_3_summary.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1bclean_fold_3_scale_params.json

Métricas del fold:
- n: 165
- mae: 5.802517407196486
- rmse: 7.145146288673982
- bias: 0.4221617317483306
- r: 0.528356940425921
- r2: 0.2791610564962402
- coverage_90: 0.8363636363636363
- width_90_mean: 33.646620310408096
- wis_90: 39.67411016955829
- model: M1b_clean
- fold: fold_3
- train_years: 2020,2021,2022,2023
- test_years: 2024

VALIDACIÓN TEMPORAL M1b-clean COMPLETADA
- métricas consolidadas : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1bclean_metrics_folds.csv
- predicciones consolidadas : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1bclean_pred_test_all_folds.csv

Resumen final de métricas M1b-clean:


,n,mae,rmse,bias,r,r2,coverage_90,width_90_mean,wis_90,model,fold,train_years,test_years
0,165,4.596563,5.967040,3.374788,0.663303,0.439971,1.000000,46.074545,46.074545,M1b_clean,fold_1,"2020,2021",2022
1,180,3.487976,4.361790,1.561506,0.771983,0.595958,0.983333,39.319265,39.616349,M1b_clean,fold_2,"2020,2021,2022",2023
2,165,5.802517,7.145146,0.422162,0.528357,0.279161,0.836364,33.646620,39.674110,M1b_clean,fold_3,"2020,2021,2022,2023",2024



Comparación M1b vs M1b-clean guardada en:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_compare_M1b_vs_M1bclean.csv

Tabla comparativa:


,model,fold,n,mae,rmse,bias,r,r2,coverage_90,width_90_mean,wis_90,train_years,test_years
0,M1b,fold_1,165,5.131715,7.064044,4.057671,0.650903,0.423675,1.000000,48.205675,48.205675,"2020,2021",2022
1,M1b_clean,fold_1,165,4.596563,5.967040,3.374788,0.663303,0.439971,1.000000,46.074545,46.074545,"2020,2021",2022
2,M1b,fold_2,180,3.331843,4.197950,1.411280,0.784936,0.616125,0.983333,40.585155,40.879425,"2020,2021,2022",2023
3,M1b_clean,fold_2,180,3.487976,4.361790,1.561506,0.771983,0.595958,0.983333,39.319265,39.616349,"2020,2021,2022",2023
4,M1b,fold_3,165,5.729405,7.120184,0.420367,0.532644,0.283710,0.818182,33.703959,40.217879,"2020,2021,2022,2023",2024
5,M1b_clean,fold_3,165,5.802517,7.145146,0.422162,0.528357,0.279161,0.836364,33.646620,39.674110,"2020,2021,2022,2023",2024


In [ ]:
# %%
# =========================
# CELDA 9E) Validación temporal HBM limpio
# =========================

all_metrics_m1bclean = []
all_test_preds_m1bclean = []

DRAWS_C = 1000
TUNE_C = 1500
CHAINS_C = 4
TARGET_ACCEPT_C = 0.99
MAX_TREEDEPTH_C = 15
RANDOM_SEED_C = 42

for fold_name, fold_info in FOLDS.items():
    print("\n" + "=" * 80)
    print(f"Corriendo M1b-clean - {fold_name}")
    print("Train years:", fold_info["train_years"])
    print("Test years :", fold_info["test_years"])

    # -------------------------------------------------
    # 1) Escalar covariables limpias usando solo train
    # -------------------------------------------------
    grid_scaled_c, scale_params_c = scale_grid_by_train_years(
        grid_df=grid_base_clean,
        x_cols=X_COLS_PM25_CLEAN,
        train_years=fold_info["train_years"]
    )

    # -------------------------------------------------
    # 2) Llevar covariables escaladas al panel observado limpio
    # -------------------------------------------------
    obs_scaled_c = merge_scaled_covariates_to_obs(
        obs_df=obs_panel_clean,
        grid_scaled=grid_scaled_c,
        x_cols=X_COLS_PM25_CLEAN
    )

    # -------------------------------------------------
    # 3) Ajustar modelo del fold
    # -------------------------------------------------
    idata_fold_c, test_pred_fold_c, met_fold_c = fit_hbm_m1b_fold(
        obs_scaled=obs_scaled_c,
        grid_scaled=grid_scaled_c,
        w_edges_df=w_edges,
        x_cols_raw=X_COLS_PM25_CLEAN,
        train_years=fold_info["train_years"],
        test_years=fold_info["test_years"],
        n_cells=len(cell_ids),
        n_times=len(time_ids),
        draws=DRAWS_C,
        tune=TUNE_C,
        chains=CHAINS_C,
        target_accept=TARGET_ACCEPT_C,
        max_treedepth=MAX_TREEDEPTH_C,
        random_seed=RANDOM_SEED_C,
    )

    # -------------------------------------------------
    # 4) Guardar predicciones del fold
    # -------------------------------------------------
    test_pred_fold_c["fold"] = fold_name
    pred_fold_path_c = OUT_DIR / f"HBM_PM25_M1bclean_{fold_name}_pred_test.csv"
    test_pred_fold_c.to_csv(pred_fold_path_c, index=False)

    # -------------------------------------------------
    # 5) Guardar summary del posterior
    # -------------------------------------------------
    summary_fold_c = az.summary(
        idata_fold_c,
        var_names=["alpha", "beta", "sigma_y", "tau_phi", "sigma_t"],
        round_to=4
    )
    summary_fold_path_c = OUT_DIR / f"HBM_PM25_M1bclean_{fold_name}_summary.csv"
    summary_fold_c.to_csv(summary_fold_path_c)

    # -------------------------------------------------
    # 6) Guardar parámetros de escalamiento
    # -------------------------------------------------
    scale_fold_path_c = OUT_DIR / f"HBM_PM25_M1bclean_{fold_name}_scale_params.json"
    with open(scale_fold_path_c, "w", encoding="utf-8") as f:
        json.dump(scale_params_c, f, ensure_ascii=False, indent=2)

    # -------------------------------------------------
    # 7) Consolidar métricas
    # -------------------------------------------------
    met_fold_c["model"] = "M1b_clean"
    met_fold_c["fold"] = fold_name
    met_fold_c["train_years"] = ",".join(map(str, fold_info["train_years"]))
    met_fold_c["test_years"] = ",".join(map(str, fold_info["test_years"]))

    all_metrics_m1bclean.append(met_fold_c)
    all_test_preds_m1bclean.append(test_pred_fold_c)

    print("\nGuardado del fold:")
    print("-", pred_fold_path_c)
    print("-", summary_fold_path_c)
    print("-", scale_fold_path_c)

    print("\nMétricas del fold:")
    for k, v in met_fold_c.items():
        print(f"- {k}: {v}")

# -------------------------------------------------
# 8) Consolidar resultados M1b-clean
# -------------------------------------------------
metrics_m1bclean_df = pd.DataFrame(all_metrics_m1bclean)
preds_m1bclean_df = pd.concat(all_test_preds_m1bclean, ignore_index=True)

metrics_m1bclean_path = OUT_DIR / "HBM_PM25_M1bclean_metrics_folds.csv"
preds_m1bclean_path = OUT_DIR / "HBM_PM25_M1bclean_pred_test_all_folds.csv"

metrics_m1bclean_df.to_csv(metrics_m1bclean_path, index=False)
preds_m1bclean_df.to_csv(preds_m1bclean_path, index=False)

print("\n" + "=" * 80)
print("VALIDACIÓN TEMPORAL M1b-clean COMPLETADA")
print("- métricas consolidadas :", metrics_m1bclean_path)
print("- predicciones consolidadas :", preds_m1bclean_path)

print("\nResumen final de métricas M1b-clean:")
display(metrics_m1bclean_df)

# -------------------------------------------------
# 9) Comparar M1b vs M1b-clean
# -------------------------------------------------
m1b_metrics_path = OUT_DIR / "HBM_PM25_M1b_metrics_folds.csv"

if m1b_metrics_path.exists():
    metrics_m1b_df = pd.read_csv(m1b_metrics_path).copy()
    metrics_m1b_df["model"] = "M1b"

    cols_keep = [
        "model", "fold", "n", "mae", "rmse", "bias", "r", "r2",
        "coverage_90", "width_90_mean", "wis_90", "train_years", "test_years"
    ]

    compare_clean_df = pd.concat(
        [
            metrics_m1b_df[cols_keep],
            metrics_m1bclean_df[cols_keep]
        ],
        ignore_index=True
    )

    compare_clean_path = OUT_DIR / "HBM_PM25_compare_M1b_vs_M1bclean.csv"
    compare_clean_df.to_csv(compare_clean_path, index=False)

    print("\nComparación M1b vs M1b-clean guardada en:")
    print("-", compare_clean_path)

    print("\nTabla comparativa:")
    display(compare_clean_df.sort_values(["fold", "model"]).reset_index(drop=True))
else:
    print("\nNo se encontró el archivo de métricas de M1b previo.")
    print("Se omitió la comparación M1b vs M1b-clean.")


Corriendo M1b-clean - fold_1
Train years: [2020, 2021]
Test years : [2022]


Initializing NUTS using adapt_diag...
ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: local_subtensor_merge
ERROR (pytensor.graph.rewriting.basic): node: Subtensor{i}(Subtensor{start:}.0, 0)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 1920, in process_node
    replacements = node_rewriter.transform(
                   ^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 993, in transform
    return self.fn(fgraph, node)
           ^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 416, in local_subtensor_merge
    merge_two_slices(
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 865, in

Output()

Sampling 4 chains for 1_500 tune and 1_000 draw iterations (6_000 + 4_000 draws total) took 714 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



Guardado del fold:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1bclean_fold_1_pred_test.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1bclean_fold_1_summary.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1bclean_fold_1_scale_params.json

Métricas del fold:
- n: 165
- mae: 4.596562896886717
- rmse: 5.967040210701766
- bias: 3.374788310318431
- r: 0.6633033660934421
- r2: 0.4399713554708909
- coverage_90: 1.0
- width_90_mean: 46.074545443724325
- wis_90: 46.074545443724325
- model: M1b_clean
- fold: fold_1
- train_years: 2020,2021
- test_years: 2022

Corriendo M1b-clean - fold_2
Train years: [2020, 2021, 2022]
Test years : [2023]


Initializing NUTS using adapt_diag...
ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: local_subtensor_merge
ERROR (pytensor.graph.rewriting.basic): node: Subtensor{i}(Subtensor{start:}.0, 0)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 1920, in process_node
    replacements = node_rewriter.transform(
                   ^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 993, in transform
    return self.fn(fgraph, node)
           ^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 416, in local_subtensor_merge
    merge_two_slices(
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 865, in

Output()

# CELDA 9F — Comparación de diagnósticos de convergencia: M1b vs M1b-clean

 **Objetivo:**
 verificar si la limpieza de covariables no solo mejoró las métricas predictivas,
 sino también la estabilidad bayesiana del muestreo.

 **Qué hace esta celda:**
 1. Lee los archivos `summary.csv` de:
    - `M1b`
    - `M1b_clean`
 2. Extrae por fold:
    - `max_r_hat`
    - `min_ess_bulk`
    - `min_ess_tail`
 3. Consolida la comparación.
 4. Marca reglas simples:
    - `r_hat_ok`: max r_hat <= 1.01
    - `ess_bulk_ok`: min ess_bulk >= 400
    - `ess_tail_ok`: min ess_tail >= 400

 **Interpretación esperada:**
 si `M1b_clean` mantiene o mejora estos diagnósticos, será el modelo elegido
 para el ajuste final sobre todo el período.

In [18]:
# %%
# =========================
# CELDA 9F) Diagnósticos M1b vs M1b-clean
# =========================

summary_files_clean_compare = {
    "M1b": {
        "fold_1": OUT_DIR / "HBM_PM25_M1b_fold_1_summary.csv",
        "fold_2": OUT_DIR / "HBM_PM25_M1b_fold_2_summary.csv",
        "fold_3": OUT_DIR / "HBM_PM25_M1b_fold_3_summary.csv",
    },
    "M1b_clean": {
        "fold_1": OUT_DIR / "HBM_PM25_M1bclean_fold_1_summary.csv",
        "fold_2": OUT_DIR / "HBM_PM25_M1bclean_fold_2_summary.csv",
        "fold_3": OUT_DIR / "HBM_PM25_M1bclean_fold_3_summary.csv",
    }
}

rows_diag_clean = []

for model_name, model_files in summary_files_clean_compare.items():
    for fold_name, path_summary in model_files.items():
        if not path_summary.exists():
            print(f"No existe: {path_summary}")
            continue

        df_sum = pd.read_csv(path_summary, index_col=0)

        needed_cols = ["r_hat", "ess_bulk", "ess_tail"]
        for c in needed_cols:
            if c not in df_sum.columns:
                raise ValueError(f"Falta la columna '{c}' en {path_summary.name}")

        row = {
            "model": model_name,
            "fold": fold_name,
            "max_r_hat": float(df_sum["r_hat"].max()),
            "min_ess_bulk": float(df_sum["ess_bulk"].min()),
            "min_ess_tail": float(df_sum["ess_tail"].min()),
        }

        row["r_hat_ok"] = row["max_r_hat"] <= 1.01
        row["ess_bulk_ok"] = row["min_ess_bulk"] >= 400
        row["ess_tail_ok"] = row["min_ess_tail"] >= 400

        rows_diag_clean.append(row)

diag_clean_compare = (
    pd.DataFrame(rows_diag_clean)
    .sort_values(["fold", "model"])
    .reset_index(drop=True)
)

diag_clean_compare_path = OUT_DIR / "HBM_PM25_compare_diagnostics_M1b_vs_M1bclean.csv"
diag_clean_compare.to_csv(diag_clean_compare_path, index=False)

print("Diagnósticos comparativos guardados en:")
print("-", diag_clean_compare_path)

print("\nTabla comparativa de convergencia:")
display(diag_clean_compare)

Diagnósticos comparativos guardados en:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_compare_diagnostics_M1b_vs_M1bclean.csv

Tabla comparativa de convergencia:


,model,fold,max_r_hat,min_ess_bulk,min_ess_tail,r_hat_ok,ess_bulk_ok,ess_tail_ok
0,M1b,fold_1,1.0017,1091.8871,1808.5574,True,True,True
1,M1b_clean,fold_1,1.0042,1016.4907,1839.4567,True,True,True
2,M1b,fold_2,1.0043,874.8228,1758.3279,True,True,True
3,M1b_clean,fold_2,1.0037,667.5744,1724.5314,True,True,True
4,M1b,fold_3,1.0021,429.2079,899.2738,True,True,True
5,M1b_clean,fold_3,1.0085,410.1968,949.1921,True,True,True


# CELDA 10 — Ajuste final del modelo seleccionado M1b-clean

 **Objetivo:**
 ajustar el modelo final seleccionado para PM2.5 usando todas las
 observaciones válidas de 2020–2024 y las covariables limpias.

 **Modelo final seleccionado:**
 - observación: `PM25_obs`
 - espacial: `ICAR`
 - temporal: `RW1`
 - covariables limpias:
 - `Vel_viento_idw_clean`
 - `diff_PM25_grid_slog_clean`
 - `adv_proxy_PM25_grid_clean`

 **Qué hace esta celda:**
 1. Escala covariables limpias usando todo el período.
 2. Ajusta el modelo final `M1b_clean`.
 3. Predice sobre toda la malla 3 km y todos los meses.
 4. Exporta la superficie final:
    - `p05_hbm`
    - `p50_hbm`
    - `p95_hbm`
    - `width_90`
 5. Guarda también:
    - summary final
    - diagnósticos finales
    - parámetros de escalamiento
    - posterior en NetCDF

 **Salidas principales:**
 - `HBM_PM25_M1bclean_surface_final.csv`
 - `HBM_PM25_M1bclean_final_summary.csv`
 - `HBM_PM25_M1bclean_final_diagnostics.csv`
 - `HBM_PM25_M1bclean_final_scale_params.json`
 - `HBM_PM25_M1bclean_final_posterior.nc`

In [19]:
# %%
# =========================
# CELDA 10) Ajuste final M1b-clean
# =========================

ALL_YEARS = [2020, 2021, 2022, 2023, 2024]

DRAWS_FINAL_C = 1200
TUNE_FINAL_C = 1800
CHAINS_FINAL_C = 4
TARGET_ACCEPT_FINAL_C = 0.99
MAX_TREEDEPTH_FINAL_C = 15
RANDOM_SEED_FINAL_C = 42

# -------------------------------------------------
# 1) Escalar covariables limpias con todo el período
# -------------------------------------------------
grid_scaled_all_c, scale_params_all_c = scale_grid_by_train_years(
    grid_df=grid_base_clean,
    x_cols=X_COLS_PM25_CLEAN,
    train_years=ALL_YEARS
)

obs_scaled_all_c = merge_scaled_covariates_to_obs(
    obs_df=obs_panel_clean,
    grid_scaled=grid_scaled_all_c,
    x_cols=X_COLS_PM25_CLEAN
)

z_cols_c = [f"{c}_z" for c in X_COLS_PM25_CLEAN]

# -------------------------------------------------
# 2) Arrays de entrenamiento completo
# -------------------------------------------------
X_all_c = obs_scaled_all_c[z_cols_c].to_numpy(dtype=float)
y_all_raw_c = obs_scaled_all_c["PM25_obs"].to_numpy(dtype=float)
y_all_c = np.log(y_all_raw_c + EPS) if USE_LOG else y_all_raw_c

cell_all_c = obs_scaled_all_c["cell_idx"].to_numpy(dtype=int)
time_all_c = obs_scaled_all_c["time_idx"].to_numpy(dtype=int)

ei = w_edges["i"].to_numpy(dtype=int)
ej = w_edges["j"].to_numpy(dtype=int)

p_c = X_all_c.shape[1]
alpha_mu_all_c = float(np.mean(y_all_c))

print("Ajustando modelo final M1b-clean...")
print("- observaciones válidas:", len(obs_scaled_all_c))
print("- estaciones únicas    :", obs_scaled_all_c["Estacion"].nunique())
print("- celdas observadas     :", obs_scaled_all_c["cell_id"].nunique())
print("- celdas totales malla  :", len(cell_ids))
print("- meses totales         :", len(time_ids))
print("- covariables usadas    :", X_COLS_PM25_CLEAN)

# -------------------------------------------------
# 3) Ajuste final M1b-clean
# -------------------------------------------------
with pm.Model() as final_model_m1bclean:
    # efectos fijos regularizados
    alpha = pm.Normal("alpha", mu=alpha_mu_all_c, sigma=1.0)
    beta = pm.Normal("beta", mu=0.0, sigma=0.5, shape=p_c)

    # error observacional
    sigma_y = pm.HalfNormal("sigma_y", sigma=0.75)

    # espacial ICAR
    tau_phi = pm.Exponential("tau_phi", 2.0)
    phi_raw = pm.Normal("phi_raw", mu=0.0, sigma=1.0, shape=len(cell_ids))
    phi = pm.Deterministic("phi", phi_raw - pt.mean(phi_raw))
    pm.Potential("icar_penalty", -0.5 * tau_phi * pt.sum((phi[ei] - phi[ej]) ** 2))

    # temporal RW1
    sigma_t = pm.HalfNormal("sigma_t", sigma=0.25)
    delta_raw = pm.GaussianRandomWalk("delta_raw", sigma=sigma_t, shape=len(time_ids))
    delta = pm.Deterministic("delta", delta_raw - pt.mean(delta_raw))

    # media
    mu_all_c = alpha + pt.dot(X_all_c, beta) + phi[cell_all_c] + delta[time_all_c]

    # likelihood
    pm.Normal("y_obs", mu=mu_all_c, sigma=sigma_y, observed=y_all_c)

    # muestreo
    idata_final_m1bclean = pm.sample(
        draws=DRAWS_FINAL_C,
        tune=TUNE_FINAL_C,
        chains=CHAINS_FINAL_C,
        init="adapt_diag",
        target_accept=TARGET_ACCEPT_FINAL_C,
        max_treedepth=MAX_TREEDEPTH_FINAL_C,
        random_seed=RANDOM_SEED_FINAL_C,
        return_inferencedata=True,
        progressbar=True,
    )

# -------------------------------------------------
# 4) Predicción sobre toda la malla y todos los meses
# -------------------------------------------------
X_grid_all_c = grid_scaled_all_c[z_cols_c].to_numpy(dtype=float)
cell_grid_all_c = grid_scaled_all_c["cell_idx"].to_numpy(dtype=int)
time_grid_all_c = grid_scaled_all_c["time_idx"].to_numpy(dtype=int)

p05_grid_c, p50_grid_c, p95_grid_c = posterior_predict_concentration(
    idata=idata_final_m1bclean,
    X_mat=X_grid_all_c,
    cell_idx_arr=cell_grid_all_c,
    time_idx_arr=time_grid_all_c,
    eps=EPS
)

surface_final_m1bclean = grid_scaled_all_c[
    ["cell_id", "fecha", "year", "month", "cell_idx", "time_idx"]
].copy()

surface_final_m1bclean["p05_hbm"] = p05_grid_c
surface_final_m1bclean["p50_hbm"] = p50_grid_c
surface_final_m1bclean["p95_hbm"] = p95_grid_c
surface_final_m1bclean["width_90"] = (
    surface_final_m1bclean["p95_hbm"] - surface_final_m1bclean["p05_hbm"]
)

# -------------------------------------------------
# 5) Summary y diagnósticos finales
# -------------------------------------------------
summary_final_m1bclean = az.summary(
    idata_final_m1bclean,
    var_names=["alpha", "beta", "sigma_y", "tau_phi", "sigma_t"],
    round_to=4
)

diag_final_m1bclean = pd.DataFrame([{
    "model": "M1b_clean_final",
    "max_r_hat": float(summary_final_m1bclean["r_hat"].max()),
    "min_ess_bulk": float(summary_final_m1bclean["ess_bulk"].min()),
    "min_ess_tail": float(summary_final_m1bclean["ess_tail"].min()),
    "r_hat_ok": bool(summary_final_m1bclean["r_hat"].max() <= 1.01),
    "ess_bulk_ok": bool(summary_final_m1bclean["ess_bulk"].min() >= 400),
    "ess_tail_ok": bool(summary_final_m1bclean["ess_tail"].min() >= 400),
}])

# -------------------------------------------------
# 6) Guardar salidas
# -------------------------------------------------
surface_final_clean_path = OUT_DIR / "HBM_PM25_M1bclean_surface_final.csv"
summary_final_clean_path = OUT_DIR / "HBM_PM25_M1bclean_final_summary.csv"
diag_final_clean_path = OUT_DIR / "HBM_PM25_M1bclean_final_diagnostics.csv"
scale_final_clean_path = OUT_DIR / "HBM_PM25_M1bclean_final_scale_params.json"
idata_final_clean_path = OUT_DIR / "HBM_PM25_M1bclean_final_posterior.nc"

surface_final_m1bclean.to_csv(surface_final_clean_path, index=False)
summary_final_m1bclean.to_csv(summary_final_clean_path)
diag_final_m1bclean.to_csv(diag_final_clean_path, index=False)

with open(scale_final_clean_path, "w", encoding="utf-8") as f:
    json.dump(scale_params_all_c, f, ensure_ascii=False, indent=2)

az.to_netcdf(idata_final_m1bclean, idata_final_clean_path)

# -------------------------------------------------
# 7) Resumen final
# -------------------------------------------------
print("\nAJUSTE FINAL M1b-clean COMPLETADO")
print("- superficie final     :", surface_final_clean_path)
print("- summary final        :", summary_final_clean_path)
print("- diagnósticos finales :", diag_final_clean_path)
print("- scale params         :", scale_final_clean_path)
print("- posterior netcdf     :", idata_final_clean_path)

print("\nDiagnósticos finales:")
display(diag_final_m1bclean)

print("\nPrimeras filas de la superficie final limpia:")
display(surface_final_m1bclean.head())

Ajustando modelo final M1b-clean...
- observaciones válidas: 808
- estaciones únicas    : 16
- celdas observadas     : 14
- celdas totales malla  : 254
- meses totales         : 60
- covariables usadas    : ['Vel_viento_idw_clean', 'diff_PM25_grid_slog_clean', 'adv_proxy_PM25_grid_clean']


Initializing NUTS using adapt_diag...
ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: local_subtensor_merge
ERROR (pytensor.graph.rewriting.basic): node: Subtensor{i}(Subtensor{start:}.0, 0)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 1920, in process_node
    replacements = node_rewriter.transform(
                   ^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 993, in transform
    return self.fn(fgraph, node)
           ^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 416, in local_subtensor_merge
    merge_two_slices(
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 865, in

Output()

Sampling 4 chains for 1_800 tune and 1_200 draw iterations (7_200 + 4_800 draws total) took 3413 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



AJUSTE FINAL M1b-clean COMPLETADO
- superficie final     : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1bclean_surface_final.csv
- summary final        : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1bclean_final_summary.csv
- diagnósticos finales : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1bclean_final_diagnostics.csv
- scale params         : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1bclean_final_scale_params.json
- posterior netcdf     : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1bclean_final_posterior.nc

Diagnósticos finales:


,model,max_r_hat,min_ess_bulk,min_ess_tail,r_hat_ok,ess_bulk_ok,ess_tail_ok
0,M1b_clean_final,1.0127,526.8016,1007.3885,False,True,True



Primeras filas de la superficie final limpia:


,cell_id,fecha,year,month,cell_idx,time_idx,p05_hbm,p50_hbm,p95_hbm,width_90
0,0,2020-01-01,2020,1,0,0,2.920127,15.982646,79.866240,76.946113
1,1,2020-01-01,2020,1,1,0,2.766356,15.824725,83.216746,80.450390
2,2,2020-01-01,2020,1,2,0,2.747029,15.564855,84.297618,81.550589
3,41,2020-01-01,2020,1,3,0,2.686313,15.896616,84.299508,81.613195
4,42,2020-01-01,2020,1,4,0,3.039208,15.262516,83.141450,80.102242



# CELDA 10B — Diagnóstico focal del summary final del modelo limpio

 **Objetivo:**
 identificar qué parámetro(s) del modelo final `M1b_clean` están
 generando el `max_r_hat` más alto y revisar si el problema es puntual
 o extendido.

 **Qué hace esta celda:**
 1. Lee `HBM_PM25_M1bclean_final_summary.csv`.
 2. Ordena el summary por `r_hat` de mayor a menor.
 3. Marca parámetros con:
    - `r_hat > 1.01`
    - `ess_bulk < 400`
    - `ess_tail < 400`
 4. Muestra:
    - top 20 parámetros con peor `r_hat`
    - subconjunto de parámetros problemáticos
 5. Guarda una tabla de diagnóstico focal.

 **Interpretación esperada:**
 - si solo 1 o pocos parámetros quedan apenas por encima de 1.01,
   el modelo sigue siendo razonablemente usable;
 - si son muchos, habría que volver a ajustar.

In [9]:
# %%
# =========================
# CELDA 10B) Diagnóstico focal del summary final limpio
# =========================

summary_final_clean_path = OUT_DIR / "HBM_PM25_M1bclean_final_summary.csv"
summary_final_clean = pd.read_csv(summary_final_clean_path, index_col=0)

needed_cols = ["mean", "sd", "ess_bulk", "ess_tail", "r_hat"]
missing_cols = [c for c in needed_cols if c not in summary_final_clean.columns]
if missing_cols:
    raise ValueError(f"Faltan columnas en el summary final: {missing_cols}")

summary_diag = summary_final_clean.copy().reset_index().rename(columns={"index": "parametro"})

summary_diag["flag_rhat"] = summary_diag["r_hat"] > 1.01
summary_diag["flag_ess_bulk"] = summary_diag["ess_bulk"] < 400
summary_diag["flag_ess_tail"] = summary_diag["ess_tail"] < 400

summary_diag["n_flags"] = (
    summary_diag["flag_rhat"].astype(int) +
    summary_diag["flag_ess_bulk"].astype(int) +
    summary_diag["flag_ess_tail"].astype(int)
)

# top por rhat
top_rhat = summary_diag.sort_values(["r_hat", "ess_bulk"], ascending=[False, True]).head(20).copy()

# parámetros problemáticos
problem_params = summary_diag.loc[
    (summary_diag["flag_rhat"]) |
    (summary_diag["flag_ess_bulk"]) |
    (summary_diag["flag_ess_tail"])
].sort_values(["n_flags", "r_hat", "ess_bulk"], ascending=[False, False, True]).copy()

# guardar
summary_diag_path = OUT_DIR / "HBM_PM25_M1bclean_final_summary_diagnostic_focus.csv"
summary_diag.to_csv(summary_diag_path, index=False)

print("Archivo guardado:")
print("-", summary_diag_path)

print("\nResumen global del summary final:")
print("- número total de parámetros        :", len(summary_diag))
print("- parámetros con r_hat > 1.01       :", int(summary_diag["flag_rhat"].sum()))
print("- parámetros con ess_bulk < 400     :", int(summary_diag["flag_ess_bulk"].sum()))
print("- parámetros con ess_tail < 400     :", int(summary_diag["flag_ess_tail"].sum()))

print("\nTop 20 parámetros con mayor r_hat:")
display(top_rhat[["parametro", "mean", "sd", "ess_bulk", "ess_tail", "r_hat", "n_flags"]])

print("\nParámetros problemáticos:")
display(problem_params[["parametro", "mean", "sd", "ess_bulk", "ess_tail", "r_hat", "flag_rhat", "flag_ess_bulk", "flag_ess_tail"]])

Archivo guardado:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1bclean_final_summary_diagnostic_focus.csv

Resumen global del summary final:
- número total de parámetros        : 7
- parámetros con r_hat > 1.01       : 1
- parámetros con ess_bulk < 400     : 0
- parámetros con ess_tail < 400     : 0

Top 20 parámetros con mayor r_hat:


,parametro,mean,sd,ess_bulk,ess_tail,r_hat,n_flags
0,alpha,2.6379,0.2511,526.8016,1007.3885,1.0127,1
4,sigma_y,0.2032,0.0054,7440.3058,2743.8278,1.0037,0
3,beta[2],0.0091,0.0105,10919.7551,3657.9062,1.0029,0
1,beta[0],-0.1273,0.0311,9711.4607,3493.5944,1.0022,0
5,tau_phi,0.0013,0.0013,6005.1881,2698.3827,1.0009,0
2,beta[1],-0.0282,0.0053,10813.2805,3000.0797,1.0009,0
6,sigma_t,0.2462,0.0248,10397.5674,3142.9329,1.0006,0



Parámetros problemáticos:


,parametro,mean,sd,ess_bulk,ess_tail,r_hat,flag_rhat,flag_ess_bulk,flag_ess_tail
0,alpha,2.6379,0.2511,526.8016,1007.3885,1.0127,True,False,False
